# Optimización de Modelos Tabulares para competencia "PetFinder Adoption Prediction"
#### Maestría en Ciencia de Datos - Cohorte 25-26


### **Asignatura:** Laboratorio de Implementación II
### **Grupo:** Nº 8
#### **Profesores:**
* Andrés Araneo
* Tomás Lanza
#### **Alumnos Grupo 8:**
* Analía Ale
* Diego Farfán
* Jaime López Garrido
* Gabriel Martina
* Lucía Pereyra Huertas


---
# Sección 1. Introduccion

El presente notebook desarrolla un flujo de trabajo integral para la competencia PetFinder.my Adoption Prediction. El objetivo es predecir la velocidad de adopción de mascotas mediante un pipeline que abarca desde el análisis exploratorio hasta la optimización de hiperparámetros. Se utiliza el algoritmo LightGBM optimizado con Optuna (Optimización Bayesiana) y validado mediante un esquema Stratified K-Fold de 5 particiones.

## 1.1 Estructura de Entorno y Directorios
El código está diseñado para la creación automática de subcarpetas en el directorio work/, asegurando la trazabilidad de los experimentos.

    Labo2_grupo_8/
    ├── EDA&Tabulares_v3.ipynb (Notebook principal)
    ├── EDAdashboard.py
    ├── src/ani_lab/viz.py
    ├── input/
    │   └── train/
    │       ├── train.csv (Dataset principal)
    │       └── *.csv (Tablas de referencia: Breeds, Colors, States)
    └── work/ (Generado automáticamente)
        ├── optuna_artifacts/      (Base de datos SQLite y modelos persistentes)
        └── optuna_temp_artifacts/ (Archivos temporales .joblib por cada trial)

---
# Seccion 2. Importación de librerías, configuración y carga de datos

In [ ]:
# 1. LIBRERÍAS DE SISTEMA Y MANEJO DE DATOS
import os
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from joblib import load, dump

# 2. VISUALIZACIÓN DE DATOS
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from plotly import express as px

# Importación de utilidades personalizadas (asegurar que el archivo utils.py exista)
#try:
from ani_lab.viz import plot_confusion_matrix
#except ImportError:
#    print("Aviso: 'plot_confusion_matrix' no pudo cargarse desde 'utils'.")

# 3. MACHINE LEARNING Y EVALUACIÓN
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score
from sklearn.utils import shuffle

# 4. OPTIMIZACIÓN DE HIPERPARÁMETROS (OPTUNA)
import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

# Gestión de advertencias
warnings.filterwarnings('ignore')

# Configuración visual de gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (10, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

In [ ]:

# CONFIGURACIÓN DE RUTAS Y ENTORNO 

BASE_DIR = Path.cwd() # Asumimos que el notebook se ejecuta desde la raíz del proyecto

# Rutas de Entrada
INPUT_DIR = BASE_DIR / "input" / "train"
TRAIN_PATH = INPUT_DIR / "train.csv"
BREED_LABELS_PATH = INPUT_DIR / "PetFinder-BreedLabels.csv"
COLOR_LABELS_PATH = INPUT_DIR / "PetFinder-ColorLabels.csv"
STATE_LABELS_PATH = INPUT_DIR / "PetFinder-StateLabels.csv"

# Rutas de Salida
WORK_DIR = BASE_DIR / "work"
OPTUNA_DIR = WORK_DIR / "optuna_artifacts"
# Agregamos la ruta para los archivos  de Optuna
OPTUNA_TEMP_DIR = WORK_DIR / "optuna_temp_artifacts"
PATH_TO_TEMP_FILES = WORK_DIR / "optuna_temp_artifacts" 
PATH_TO_OPTUNA_ARTIFACTS = WORK_DIR / "optuna_artifacts"

# Actualizamos la creación de directorios
dirs = [WORK_DIR, OPTUNA_DIR, OPTUNA_TEMP_DIR]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
TEST_SIZE = 0.2

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(SEED)
print(f"Entorno configurado. Variable PATH_TO_TEMP_FILES vinculada a: {OPTUNA_TEMP_DIR}")


### Carga de datos

Se leen 4 archivos .csv:
* train, que contiene todos los datos de mascotas adoptadas que utilizaremos para construir el set de entrenamiento y testeo
* BreedLabels, diccionario de razas que en el dataset se representan con un número
* ColorLabels, diccionario de colores que en el dataset se representan con un número
* StateLabels, diccionario de Estados que en el dataset se representan con un número


In [ ]:
# CARGA DE DATOS

def load_dataset():
    # Diccionario para mapear constantes a nombres de variables
    files_to_load = {
        "Train": TRAIN_PATH,
        "Breeds": BREED_LABELS_PATH,
        "Colors": COLOR_LABELS_PATH,
        "States": STATE_LABELS_PATH
    }

    # Verificación de integridad
    for name, path in files_to_load.items():
        if not path.exists():
            raise FileNotFoundError(f"Error Crítico: El archivo {name} no se encuentra en {path}")

    # Carga de archivos manteniendo los nombres originales
    train_df = pd.read_csv(TRAIN_PATH)
    breeds = pd.read_csv(BREED_LABELS_PATH)
    colors = pd.read_csv(COLOR_LABELS_PATH)
    states = pd.read_csv(STATE_LABELS_PATH)

    return train_df, breeds, colors, states

# Ejecución de la carga
df, breed_labels, color_labels, state_labels = load_dataset()

print(f"Carga exitosa desde: {INPUT_DIR}")
print(f"Registros: {df.shape[0]:,} | Columnas: {df.shape[1]}")
display(df.head())

---
# Sección 3: EDA (Análisis Exploratorio de Datos)

**Variable objetivo: `AdoptionSpeed`** (velocidad de adopción)
- `0` → Adoptado el mismo día
- `1` → Adoptado en 1–7 días
- `2` → Adoptado en 8–30 días
- `3` → Adoptado en 31–90 días
- `4` → No adoptado luego de 100 días

## 3.1 Decodificación de variables categóricas

Dado que el dataset trae las variables categóricas codificadas como números, para que los gráficos y tablas del EDA sean fáciles de leer se agregan columnas paralelas con los nombres reales de cada categoría, usando los diccionarios de los archivos .csv provistos y los mapeos manuales documentados en Kaggle (Type, Gender, etc.). Se mantienen las columnas originales.



In [ ]:
# Mapeos de diccionarios a partir de los archivos csv.
breed_map  = dict(zip(breed_labels['BreedID'], breed_labels['BreedName']))
color_map  = dict(zip(color_labels['ColorID'], color_labels['ColorName']))
state_map  = dict(zip(state_labels['StateID'], state_labels['StateName']))

# Mapeos manuales a partir de la información brindada en la competencia de Kaggle.
type_map         = {1: 'Perro', 2: 'Gato'}
gender_map       = {1: 'Macho', 2: 'Hembra', 3: 'Mixto'}
size_map         = {1: 'Pequeño', 2: 'Mediano', 3: 'Grande', 4: 'Extra Grande', 0: 'Sin especificar'}
fur_map          = {1: 'Corto', 2: 'Medio', 3: 'Largo', 0: 'Sin especificar'}
health_map       = {1: 'Sano', 2: 'Menor lesión', 3: 'Lesión grave'}
bool_map         = {1: 'Sí', 2: 'No', 3: 'No sabe'}  #Aplica para variables Vaccinated, Dewormed y Sterilized.
adoption_map     = {
    0: 'Mismo día',
    1: '1ª semana',
    2: '1er mes',
    3: '2do/3er mes',
    4: 'No adoptado'
}

# Se crea una columna con los valores legibles de cada categoría definidas en los diccionarios y mantiene las originales.

df['Type_label']        = df['Type'].map(type_map)
df['Gender_label']      = df['Gender'].map(gender_map)
df['MaturitySize_label']= df['MaturitySize'].map(size_map)
df['FurLength_label']   = df['FurLength'].map(fur_map)
df['Health_label']      = df['Health'].map(health_map)
df['Vaccinated_label']  = df['Vaccinated'].map(bool_map)
df['Dewormed_label']    = df['Dewormed'].map(bool_map)
df['Sterilized_label']  = df['Sterilized'].map(bool_map)
df['Breed1_label']      = df['Breed1'].map(breed_map)
df['Breed2_label']      = df['Breed2'].map(breed_map)
df['Color1_label']      = df['Color1'].map(color_map)
df['Color2_label']      = df['Color2'].map(color_map)
df['Color3_label']      = df['Color3'].map(color_map)
df['State_label']       = df['State'].map(state_map)
df['AdoptionSpeed_label']= df['AdoptionSpeed'].map(adoption_map)

print('Decodificación completa ✓')

In [ ]:
# Visualización de las columnas creadas y agregadas en el dataset.
df.head()

## 3.2 Visualización general del dataset

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 3.3 Registros duplicados
Se verifica que no haya registros duplicados en el dataset.

In [ ]:
print(f'Registros duplicados: {df.duplicated().sum()}')

## 3.4 Valores faltantes

### Columnas originales:

In [ ]:
cols_originales = [c for c in df.columns if not c.endswith('_label')]

missing = pd.DataFrame({
    'Faltantes': df[cols_originales].isnull().sum(),
    'Porcentaje (%)': (df[cols_originales].isnull().sum() / len(df) * 100).round(2)
}).sort_values('Faltantes', ascending=False)

print(f'Total de columnas: {len(cols_originales)}')
print(f'Columnas con al menos 1 faltante: {(missing["Faltantes"] > 0).sum()}\n')
print(missing.to_string())

Se observa que solo 2 variables tienen faltantes:
* Description: solo 13 observaciones, 0.09%; no se considera relevante.
* Name: con 1265 datos faltantes, representando el 8.44%; una porción importante del dataset. Además pueden existir registros con el campo completo sin un nombre real. Se procede a analizar esta variable con mayor detalle.

### Visualización de principales nombres para gatos y perros

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, tipo, label, color in zip(axes, [1, 2], ['Perros', 'Gatos'], ['steelblue', 'salmon']):
    nombres_freq = (
        df[df['Type'] == tipo]['Name']
        .value_counts()
        .head(20)
        .sort_values()
    )

    nombres_freq.plot(kind='barh', ax=ax, color=color, edgecolor='white')
    ax.set_title(f'Top 20 nombres — {label}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Frecuencia')
    ax.set_ylabel('')
    for i, v in enumerate(nombres_freq):
        ax.text(v + 0.3, i, str(v), va='center', fontsize=8)

plt.suptitle('Nombres más frecuentes por tipo de mascota', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusión:

En los gráficos de barras se ven nombres genéricos como "Puppy" o "Puppies" (para perros), y "Kittens" o "Kitty" (para gatos), o directamente "No Name". Esto evidencia que no todos los registros con Name completado contienen nombres reales. Por lo tanto, en el feature engineering se analizará qué hacer con esta variable y si darle o no un tratamiento especial a estos casos.

### Columnas "Label"
Análisis de datos faltantes en columnas creadas para asignar la leyenda a las categorías del dataset representadas con números. Si hay faltantes en estas variables significa que en esos registros la categoría asignada no corresponde a ninguna de las que se encuentran en el diccionario.

In [ ]:
cols_label = [c for c in df.columns if c.endswith('_label')]

missing = pd.DataFrame({
    'Faltantes': df[cols_label].isnull().sum(),
    'Porcentaje (%)': (df[cols_label].isnull().sum() / len(df) * 100).round(2)
}).sort_values('Faltantes', ascending=False)

print(f'Total de columnas: {len(cols_label)}')
print(f'Columnas con al menos 1 faltante: {(missing["Faltantes"] > 0).sum()}\n')
print(missing.to_string())

Se evidencian faltantes en 4 variables:

* Breed2_label: el 72% de los registros no tiene categoría de Breed2. Esto no significa necesariamente que la mascota sea de raza pura, ya que muchos registros tienen en Breed1 razas genéricas como "Mixed Breed" (perros mestizos) o "Domestic Short/Medium/Long Hair" (gatos comunes). En estos casos, la ausencia de Breed2 simplemente refleja que no hay una segunda raza que combinar. La distinción entre raza pura y raza mixta requiere considerar tanto la presencia de Breed2 como el valor concreto de Breed1, lo que se abordará en la sección 1.9 y en el feature engineering. Es correcto que estos registros tengan NA en esta variable.

* Color2_label y Color3_label: el 71% de los registros no tiene categoría de Color3 y el 29% de Color2. Esto significa que hay mascotas que tienen 1 solo color, y otras que tienen 2. Es correcto que estos registros tengan NA en estas variables. 

* Breed1_label: hay 5 registros sin categoría de Breed1. Se procede a checkear si estos registros tampoco tienen categoría de Breed2:

In [ ]:
ambos_nulos = (df['Breed1_label'].isnull() & df['Breed2_label'].isnull()).sum()
print(f'Registros con Breed1 y Breed2 nulos: {ambos_nulos}')

Conclusión:

Se validó que no existen casos con Breed1_label y Breed2_label nulos en simultáneo. Por lo tanto, toda mascota tiene al menos una raza reportada. La distinción entre raza pura y raza mixta requiere considerar tanto si Breed2 está completo como el valor específico de Breed1 (algunos valores son categorías genéricas como "Mixed Breed" o "Domestic Short Hair"). Esta es una potencial feature a construir en Feature Engineering.

## 3.5 Variable objetivo: AdoptionSpeed (velocidad de adopción)

In [ ]:
import matplotlib.colors as mcolors

cmap = mcolors.LinearSegmentedColormap.from_list('green_red', ['#2ecc71', '#f39c12', '#e74c3c'])
PALETTE_GRAD = [cmap(i / 4) for i in range(5)]

counts = df['AdoptionSpeed'].value_counts().sort_index()
labels = [adoption_map[i] for i in counts.index]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(labels, counts.values, color=PALETTE_GRAD, edgecolor='white')
ax.set_title('Distribución de AdoptionSpeed', fontsize=13, fontweight='bold')
ax.set_xlabel('Clase')
ax.set_ylabel('Cantidad')
ax.tick_params(axis='x', rotation=0)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

Conclusión:

Las clases 1 a 4 se distribuyen de forma relativamente uniforme (entre 21% y 28% cada una). La clase 0 ("Mismo día") tiene un menor porcentaje en comparación con las demás (2.7%). Sin embarso esta proporción es suficiente para no tener problemas a la hora de construir un modelo predictivo con LigthGBM.

## 3.6 Tipo de mascota (Perro vs. Gato)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribución general
type_counts = df['Type_label'].value_counts()
axes[0].bar(type_counts.index, type_counts.values,
            color=[ '#5BC8F5','#4C72B0'])
axes[0].set_ylim(0, 9000)
axes[0].set_title('Cantidad por tipo de mascota')
axes[0].set_ylabel('Cantidad')
for i, (name, val) in enumerate(type_counts.items()):
    axes[0].text(i, val + 50, f'{val:,}\n({val/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)

# AdoptionSpeed por tipo
cross = df.groupby(['Type_label', 'AdoptionSpeed']).size().unstack(fill_value=0)
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
cross = cross.reindex(['Perro', 'Gato'])
cross_pct = cross_pct.reindex(['Perro', 'Gato'])

ax = cross.plot(kind='bar', ax=axes[1], color=PALETTE_GRAD, width=0.7)
axes[1].set_title('AdoptionSpeed por tipo de mascota')
axes[1].set_xlabel('')
axes[1].set_ylabel('Cantidad')
axes[1].legend(title='AdoptionSpeed', labels=list(adoption_map.values()), fontsize=8)
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylim(0, 3000)

# Etiquetas con cantidad y proporción
for i, container in enumerate(ax.containers):
    for j, bar in enumerate(container):
        cantidad = bar.get_height()
        pct = cross_pct.iloc[j, i]
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 5,
                f'{int(cantidad):,}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=9)

plt.suptitle('Análisis por tipo de mascota', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusión:

El primer gráfico muestra que el 54.2% de los registros son de perros y 45.8% de gatos: cantidades similares, con una ligera mayoría de perros. 
El segundo gráfico refleja la distribución de la velocidad de adopción para perros y gatos, que se reparte de manera comparable entre ambos tipos de mascotas, sin que ninguno se adopte claramente más rápido que el otro según la categoría de AdoptionSpeed asignada.

## 3.7 Algunas variables categóricas

Construiremos graficos que nos permitan visualizar la frecuencia y la distribución de la velocidad de adopción en las distintas categorías de las siguientes variables categóricas:
* Tamaño en la adultez
* Largo del pelaje
* Colores
* Género
* Variables asociadas a la salud

Y presentamos en cada caso las conclusiones de los graficos.

In [ ]:
orden_categorias = {
    'MaturitySize_label': ['Pequeño', 'Mediano', 'Grande', 'Extra Grande'],
    'FurLength_label':    ['Corto', 'Medio', 'Largo'],
    'Color1_label': ['Black', 'Brown', 'Golden', 'Yellow', 'Cream', 'Gray', 'White'],
    'Color2_label': ['Black', 'Brown', 'Golden', 'Yellow', 'Cream', 'Gray', 'White'],
    'Color3_label': ['Black', 'Brown', 'Golden', 'Yellow', 'Cream', 'Gray', 'White'],
    'Gender_label':       ['Macho', 'Hembra', 'Mixto'],
    'Vaccinated_label':   ['Sí', 'No', 'No sabe'],
    'Dewormed_label':     ['Sí', 'No', 'No sabe'],
    'Sterilized_label':   ['Sí', 'No', 'No sabe'],
    'Health_label':       ['Sano', 'Menor lesión', 'Lesión grave'],
}

def plot_bloque(bloque, titulo, nrows, ncols, figsize):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten() if nrows * ncols > 1 else [axes]

    idx = 0
    for col, title in bloque:
        orden = orden_categorias.get(col, None)
        for tipo, label in [(1, 'Perro'), (2, 'Gato')]:
            ax = axes[idx]
            sub = df[df['Type'] == tipo]
            cross = sub.groupby([col, 'AdoptionSpeed']).size().unstack(fill_value=0)

            if orden:
                cross = cross.reindex([o for o in orden if o in cross.index])

            cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
            cross_pct.plot(kind='bar', ax=ax, color=PALETTE_GRAD, width=0.75,
                           legend=(idx == 0))
            ax.set_title(f'{title} — {label}')
            ax.set_xlabel('')
            ax.set_ylabel('% dentro de categoría')
            ax.tick_params(axis='x', rotation=0)
            if idx == 0:
                ax.legend(title='AdoptionSpeed', labels=list(adoption_map.values()),
                          fontsize=7, loc='upper right')

            for container in ax.containers:
                ax.bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=9, padding=2)
            
            freq_total = cross.sum(axis=1)
            ax2 = ax.twinx()
            ax2.plot(range(len(freq_total)), freq_total.values,
                     color='black', marker='o', linewidth=1.5,
                     markersize=4, linestyle='--', label='Frecuencia total')
            ax2.set_ylabel('Frecuencia total', fontsize=8)
            ax2.tick_params(axis='y', labelsize=10)
            ax2.set_ylim(0, freq_total.values.max() * 1.3)  # arranca en 0
            ax2.yaxis.grid(False)                            # sin cuadrícula

            idx += 1

    plt.suptitle(titulo, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


### Tamaño y pelaje

In [ ]:
# Bloque 1: Tamaño y pelaje
plot_bloque(
    [('MaturitySize_label', 'Tamaño adulto'), ('FurLength_label', 'Largo de pelaje')],
    'Tamaño y pelaje vs. AdoptionSpeed', nrows=2, ncols=2, figsize=(14, 10)
)


Conclusiones:

Tamaño:

* En ambos tipos de mascotas (perros y gatos) sucede que el tamaño "Extra Grande" tiene muy poca frecuencia.
* Para los demás tamaños, no pareciera haber una relación evidente con la velocidad de adopción. Las clases se encuentran repartidas de manera similar para cada tamaño, ya sean perros o gatos.


Largo de pelaje:

* La mayoría de las mascotas tienen pelaje corto, en segundo lugar medio, y en menor cantidad pelaje largo.
* Ya sea para perros o gatos, se visualiza que a mayor largo de pelaje, más rápido se adopta la mascota. Se observa cómo a medida que el pelaje es más largo, crecen las barras de clase 0 y 1 (adoptado el mismo día y la 1ª semana) y se achican las barras de clase 3 y 4 (adoptado entre 2do y 3er mes o sin adoptar).
* Observación: En gatos esta tendencia podría estar asociada a que los de pelo largo suelen pertenecer a razas reconocibles y buscadas (como Persas), mientras que los de pelo corto son mayormente "Domestic Short Hair" (gatos comunes). Es decir, el efecto del pelaje podría estar parcialmente confundido con el efecto del tipo de raza.

### Colores

In [ ]:
# Bloque 2: Color
for col_raw, col_label in [('Color2', 'Color2_label'), ('Color3', 'Color3_label')]:
    if col_label not in df.columns:
        df[col_label] = df[col_raw].map(color_map).fillna('Desconocido')

for col, titulo in [('Color1_label', 'Color 1'), ('Color2_label', 'Color 2'), ('Color3_label', 'Color 3')]:
    plot_bloque(
        [(col, titulo)],
        f'{titulo} vs. AdoptionSpeed', nrows=1, ncols=2, figsize=(14, 5)
    )

Conclusión - Color:

* La mayoría de las mascotas tienen colores oscuros (mayores frecuencias en negro y marrón, ya sea para gato o perro, color1 y/o color2).
* La velocidad de adopción se reparte de manera similar entre categorías, excepto algunos casos que destacan. Por ejemplo, una gran proporción de los perros de color amarillo no se adopta (42% en Color1, 36% en Color2, 47% en Color3).

Se procede a analizar si la cantidad de colores tiene alguna relación con la velocidad de adopción:

In [ ]:
# Cantidad de colores únicos por mascota

def contar_colores(row):
    colores = set()
    for col in ['Color1_label', 'Color2_label', 'Color3_label']:
        val = row[col]
        if pd.notna(val):
            colores.add(val)
    return len(colores)

df['n_colores'] = df.apply(contar_colores, axis=1)

df['ColoresCategoria'] = df['n_colores'].map({
    1: '1 color',
    2: '2 colores',
    3: '3 colores'
})

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

orden_colores = ['1 color', '2 colores', '3 colores']
verde_palette = ['#2ecc71', '#27ae60', '#1a5e37']

for ax, tipo, label in zip(axes, [1, 2], ['Perros', 'Gatos']):
    sub = df[(df['Type'] == tipo) & (df['ColoresCategoria'].notna())]
    counts = sub['ColoresCategoria'].value_counts().reindex(orden_colores, fill_value=0)

    bars = ax.bar(counts.index, counts.values, color=verde_palette, edgecolor='white')
    ax.set_title(f'Cantidad de colores por mascota — {label}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frecuencia')
    ax.set_xlabel('')
    ax.set_ylim(0, 4000)

    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                f'{v:,}\n({v/len(sub)*100:.1f}%)', ha='center', va='bottom', fontsize=11)

plt.suptitle('Diversidad de colores por mascota', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

orden_colores = ['1 color', '2 colores', '3 colores']

for ax, tipo, label in zip(axes, [1, 2], ['Perros', 'Gatos']):
    sub = df[(df['Type'] == tipo) & (df['ColoresCategoria'].notna())]
    cross = sub.groupby(['ColoresCategoria', 'AdoptionSpeed']).size().unstack(fill_value=0)
    cross = cross.reindex(orden_colores, fill_value=0)
    cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100

    cross_pct.plot(kind='bar', ax=ax, color=PALETTE_GRAD, width=0.6, legend=(tipo == 1))
    ax.set_title(f'Cantidad de colores vs. AdoptionSpeed — {label}', fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('% dentro de categoría')
    ax.tick_params(axis='x', rotation=0)
    if tipo == 1:
        ax.legend(title='AdoptionSpeed', labels=list(adoption_map.values()), fontsize=7)

    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=7, padding=2)

    freq_total = cross.sum(axis=1)
    ax2 = ax.twinx()
    ax2.plot(range(len(freq_total)), freq_total.values,
             color='black', marker='o', linewidth=1.5, markersize=4, linestyle='--')
    ax2.set_ylabel('Frecuencia total', fontsize=8)
    ax2.tick_params(axis='y', labelsize=7)
    ax2.set_ylim(0, freq_total.values.max() * 1.3)
    ax2.yaxis.grid(False)

plt.suptitle('Cantidad de colores vs. AdoptionSpeed (%)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusión:

Ya sea para perros o gatos, no pareciera que la cantidad de colores marque alguna tendencia para la velocidad de adopción.
Para las 3 categorías, la distribución de la clase objetivo es similar.

### Género

In [ ]:
# Bloque 3: Género
plot_bloque(
    [('Gender_label', 'Género')],
    'Género vs. AdoptionSpeed', nrows=1, ncols=2, figsize=(12, 5)
)


Conclusión - Género:

* Tanto como para perros como para gatos, hay mayor presencia de hembras.
* Los grupos mixtos son los que menos se adoptan. Esto probablemente se deba a que estos casos corresponden a publicaciones con varias mascotas a la vez (camadas), y no a una preferencia por género en sí.
* Para los perros, se observa que los machos se adoptan más rápidamente que las hembras (mayor clase 0 y 1 para machos, mayor clase 3 y 4 para hembras).
* Para los gatos, la distribución de clases entre machos y hembras se observa más pareja.

### Salud

In [ ]:
# Bloque 4: Salud
plot_bloque(
    [('Vaccinated_label', 'Vacunado'), ('Dewormed_label', 'Desparasitado'),
     ('Sterilized_label', 'Esterilizado'), ('Health_label', 'Salud')],
    'Salud vs. AdoptionSpeed', nrows=2, ncols=4, figsize=(20, 10)
)

Conclusión - Salud:

* En cuanto a vacunación, desparasitación y esterilización no se observan grandes diferencias en la distribución de la velocidad de adopción entre las categorías, a excepción de algunos puntos destacados, como por ejemplo:
    * Los gatos con desparasitación incierta presentan una distribución bimodal: tienen tanto la mayor proporción de adopciones inmediatas (5% en clase 0, "mismo día") como la mayor proporción de no adoptados (36% en clase 4). Es decir, se concentran más en los dos extremos del espectro que en las clases intermedias, a diferencia de los otros grupos.
    * Los perros y gatos esterilizados o con esterilización incierta tienen mayor porcentaje de no adopción. Este patrón puede estar mediado por la edad, ya que los animales adultos suelen estar esterilizados con mayor frecuencia que los cachorros.

* Lesiones: la mayoría de las mascotas son sanas. Sin embargo, se evidencia que aquellas con lesiones menores o graves tienen una alta proporción de no ser adoptadas.

## 3.8 Raza


En primer lugar se visualizan la cantidad de razas y aquellas más frecuentes para perros y gatos, respectivamente.

In [ ]:
print(f'Razas únicas en Breed1: {df["Breed1_label"].nunique()}')
print(f'Razas únicas en Breed2: {df["Breed2_label"].nunique()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, tipo, label in zip(axes, [1, 2], ['Perros', 'Gatos']):
    top_breeds = (
        df[df['Type'] == tipo]['Breed1_label']
        .value_counts()
        .head(30)
    )
    top_breeds.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Top 30 razas - {label}')
    ax.set_xlabel('Cantidad')

    for container in ax.containers:
        ax.bar_label(container, fmt='%d', label_type='edge', fontsize=10, padding=3)

plt.suptitle('Razas más frecuentes (Breed1)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, tipo, label in zip(axes, [1, 2], ['Perros', 'Gatos']):
    top_breeds = (
        df[df['Type'] == tipo]['Breed2_label']
        .value_counts()
        .head(30)
    )
    top_breeds.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Top 30 razas - {label}')
    ax.set_xlabel('Cantidad')

    for container in ax.containers:
        ax.bar_label(container, fmt='%d', label_type='edge', fontsize=10, padding=3)

plt.suptitle('Razas más frecuentes (Breed2)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusión - Razas:

* La mayoría de los perros son de "Mixed Breed" y los gatos de "Domestic Short Hair". Son categorías genéricas que no representan razas puras sino mestizos comunes. Esto es especialmente notable en gatos: las razas reconocibles (Persa, Siamés, etc.) tienen frecuencias mínimas, mientras que la enorme mayoría se concentra en las tres categorías "Domestic" (Short/Medium/Long Hair).

* Se observa que estas variables tienen una alta cardinalidad: hay muchas razas con muy pocos casos cada una, lo que dificulta su uso directo en un modelo. En el feature engineering convendrá agruparlas en menos categorías con más datos cada una.

Para sumar más información descriptiva, se creó una clasificación binaria en "Raza Pura" vs. "Raza Mixta". Se considera "Raza Pura" cuando hay una única raza específica reportada (Breed1 completo con valor no genérico, Breed2 vacío), y Raza Mixta cuando hay dos razas reportadas o cuando Breed1 corresponde a una categoría genérica (como "Mixed Breed" o "Domestic Short Hair"). Luego, se volvió a evaluar la velocidad de adopción.

In [ ]:
df['tiene_breed1'] = df['Breed1_label'].notna()
df['tiene_breed2'] = df['Breed2_label'].notna()

razas_genericas = ['Mixed Breed', 'Domestic Short Hair', 'Domestic Medium Hair', 'Domestic Long Hair']

df['TipoRaza'] = np.where(
    (df['tiene_breed1'] & df['tiene_breed2']) |
    (df['Breed1_label'].isin(razas_genericas)),
    'Raza Mixta',
    np.where(df['tiene_breed1'] & ~df['tiene_breed2'], 'Raza Pura', None)
)

# Gráfico 1: Frecuencia por tipo de raza
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tipo, label in zip(axes, [1, 2], ['Perros', 'Gatos']):
    sub = df[df['Type'] == tipo]
    categorias = {
        'Raza Pura':  (sub['tiene_breed1'] & ~sub['tiene_breed2'] &
                       ~sub['Breed1_label'].isin(razas_genericas)).sum(),
        'Raza Mixta': ((sub['tiene_breed1'] & sub['tiene_breed2']) |
                        sub['Breed1_label'].isin(razas_genericas)).sum(),
        'Ninguna':    (~sub['tiene_breed1'] & ~sub['tiene_breed2']).sum(),
    }

    bars = ax.bar(categorias.keys(), categorias.values(),
                  color=['steelblue', '#DD8452', 'salmon'], edgecolor='white')
    ax.set_title(f'Tipo de raza — {label}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frecuencia')

    for bar, v in zip(bars, categorias.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                f'{v:,}\n({v/len(sub)*100:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.suptitle('Registros por tipo de raza', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Gráfico 2: AdoptionSpeed por tipo de raza
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tipo, label in zip(axes, [1, 2], ['Perros', 'Gatos']):
    sub = df[(df['Type'] == tipo) & (df['TipoRaza'].notna())]
    cross = sub.groupby(['TipoRaza', 'AdoptionSpeed']).size().unstack(fill_value=0)
    cross = cross.reindex(['Raza Pura', 'Raza Mixta'])
    cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100

    cross_pct.plot(kind='bar', ax=ax, color=PALETTE_GRAD, width=0.6, legend=(tipo == 1))
    ax.set_title(f'Tipo de raza vs. AdoptionSpeed — {label}')
    ax.set_xlabel('')
    ax.set_ylabel('% dentro de categoría')
    ax.tick_params(axis='x', rotation=0)
    if tipo == 1:
        ax.legend(title='AdoptionSpeed', labels=list(adoption_map.values()), fontsize=7)

    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=7, padding=2)

    freq_total = cross.sum(axis=1)
    ax2 = ax.twinx()
    ax2.plot(range(len(freq_total)), freq_total.values,
             color='black', marker='o', linewidth=1.5, markersize=4, linestyle='--')
    ax2.set_ylabel('Frecuencia total', fontsize=8)
    ax2.tick_params(axis='y', labelsize=7)
    ax2.set_ylim(0, freq_total.values.max() * 1.3)
    ax2.yaxis.grid(False)

plt.suptitle('Tipo de raza vs. AdoptionSpeed (%)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusiones:

* Gráfico 1: Registros por tipo de raza

Se observa que la gran mayoría de las mascotas son de raza mixta en ambas especies: 86.5% en perros y 86.7% en gatos. Las proporciones son prácticamente idénticas entre perros y gatos, aunque las razones detrás del predominio de la raza mixta son distintas en cada caso: en perros refleja la prevalencia de mestizos reales, mientras que en gatos refleja el peso de las categorías "Domestic Short/Medium/Long Hair" (gatos comunes).


* Gráfico 2: Tipo de raza vs. AdoptionSpeed

En el caso de los perros, los de raza pura se adoptan claramente más rápido que los de raza mixta: el 31% de los puros se adopta en la primera semana frente al 16% de los mixtos, y solo el 15% de los puros queda sin adoptar frente al 32% de los mixtos. 
En cambio, para los gatos la distribución es prácticamente idéntica entre raza pura y mixta (25% sin adoptar en ambos casos, 24-25% adoptados en la primera semana en ambos casos). Esto se debe posiblemente a que la variedad de razas reales en gatos es menor: los gatos clasificados como "raza pura" pertenecen mayormente a unas pocas razas (Persa, Siamés), y la raza no es un criterio de búsqueda tan determinante para los adoptantes de gatos como sí lo es en perros.
Esta diferencia entre especies tiene implicancias para el feature engineering: la variable "Tipo de raza" será informativa para predecir la velocidad de adopción en perros, pero aportará menos en gatos.

## 3.9 Distribución geográfica (Estado)

Se analizan cuántos estados existen en el dataset y cla cantidad de registros por estado.

In [ ]:
print(f'Estados únicos: {df["State_label"].nunique()}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

state_counts = df['State_label'].value_counts().sort_values(ascending=True)
state_counts.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Cantidad de registros por estado', fontsize=13, fontweight='bold')
ax.set_xlabel('Cantidad')
ax.set_ylabel('')

for i, v in enumerate(state_counts):
    ax.text(v + 10, i, f'{v:,} ({v/len(df)*100:.1f}%)', va='center', fontsize=9)

plt.tight_layout()
plt.show()

Conclusión:

Se observa que alrededor del 84% de los registros está concentrado en solo 2 estados; Selangor y Kuala Lumpur. La variable presenta una distribución muy desbalanceada: pocos estados concentran la mayoría de los datos. Por ese motivo, se visualizan los estados que tienen al menos un 2.5% de representatividad en el dataset y todos los demás se agrupan en la categoría "Otros". Se analiza la velocidad de adopción con estas categorías.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Top 5 estados y resto agrupado en Others
top5 = df['State_label'].value_counts().head(5).index.tolist()
df['State_group'] = df['State_label'].apply(lambda x: x if x in top5 else 'Others')

# Orden: top 5 de mayor a menor + Others al final
orden_states = top5 + ['Others']

cross_state = df.groupby(['State_group', 'AdoptionSpeed']).size().unstack(fill_value=0)
cross_state = cross_state.reindex(orden_states)
cross_state_pct = cross_state.div(cross_state.sum(axis=1), axis=0) * 100

cross_state_pct.plot(kind='bar', ax=ax, color=PALETTE_GRAD, width=0.6)
ax.set_title('AdoptionSpeed por estado — Top 5 + Others')
ax.set_xlabel('')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Clase', labels=list(adoption_map.values()), fontsize=7)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=9, padding=2)

freq_total = cross_state.sum(axis=1)
ax2 = ax.twinx()
ax2.plot(range(len(freq_total)), freq_total.values,
         color='black', marker='o', linewidth=1.5, markersize=4, linestyle='--')
ax2.set_ylabel('Frecuencia total', fontsize=8)
ax2.tick_params(axis='y', labelsize=7)
ax2.set_ylim(0, freq_total.values.max() * 1.3)
ax2.yaxis.grid(False)

plt.tight_layout()
plt.show()

Conclusión - Distribución geográfica por estado: 

* Se observa que cada estado tiene su propio patrón de velocidad de adopción. 
* La categoría "Otros" (que agrupa estados con baja representatividad) presenta el mayor porcentaje de no adoptados, aunque conviene tener presente que solo concentra el aproximadamente el 16% del dataset: el comportamiento global está dominado por Selangor y Kuala Lumpur (aproximadamente el 84% de los registros).
* Se observa que cada Estado  tiene su patrón de velocidad de adopción. Y en particular, todos los agrupados en la categoría "Otros" son los que presentan el mayor porcentaje de No Adoptados.

## 3.10 Variables numéricas

Se observa en el dataset que las siguientes variables son numéricas:
['Age', 'Quantity', 'Fee', 'PhotoAmt', 'VideoAmt']

Se analiza gráficamente cada una de ellas y se extraen conclusiones al respecto.

Como puntos generales para todas las variables se observa que:
* Tienen distribución asimétrica positiva (en distinto grado): las mayores frecuencias se concentran en los menores valores, ya sea de edad, cantidad, cuota, cantidad de fotos o videos.
* Presentan valores outliers (evidenciados en los boxplots), donde la mayor cantidad de valores se concentra en un rango pequeño (o incluso en un único valor) y aparecen múltiples registros con valores aislados alejados de la mediana.

### Age (Edad)

In [ ]:
# Age
col = 'Age'
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

n, bins, patches = axes[0].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', align='mid')
axes[0].set_title(f'{col} — Distribución\nMedia: {df[col].mean():.1f} | Mediana: {df[col].median():.1f}')
axes[0].set_xlabel(col)
axes[0].set_ylabel('Frecuencia')
total = df[col].dropna().shape[0]
for patch, freq in zip(patches, n):
    if freq > 0:
        axes[0].text(patch.get_x() + patch.get_width() / 2, freq + total * 0.005,
                     f'{int(freq):,}', ha='center', va='bottom', fontsize=10)

data_by_class = [df[df['AdoptionSpeed'] == cls][col].dropna() for cls in range(5)]
bp = axes[1].boxplot(data_by_class, patch_artist=True, medianprops=dict(color='black', linewidth=2), showfliers=True)
for patch, color in zip(bp['boxes'], PALETTE_GRAD):
    patch.set_facecolor(color)
axes[1].set_xticklabels(['0', '1', '2', '3', '4'])
axes[1].set_xlabel('AdoptionSpeed')
axes[1].set_title(f'{col} por clase de AdoptionSpeed')

plt.suptitle(f'Variable: {col}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


Conclusión - Edad:

* La mayoría de las mascotas son cachorros: aproximadamente el 72% del dataset (10.804 registros) tiene edad menor a aprox. 10 meses, con mediana de 3 meses. La media (10.5 meses) está sesgada hacia arriba por la presencia de outliers de mascotas muy ancianas.
* En los boxplots se observa poca variabilidad de edad dentro de cada clase: las cajas son angostas y todas se concentran en valores bajos. La tendencia "a mayor edad, más lento se adopta" se ve más en los outliers (animales adultos/ancianos que caen en clases 3 y 4) que en las diferencias entre medianas.

### Quantity (Cantidad de mascotas por registro)

In [ ]:
# Quantity
col = 'Quantity'
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

n, bins, patches = axes[0].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', align='mid')
axes[0].set_title(f'{col} — Distribución\nMedia: {df[col].mean():.1f} | Mediana: {df[col].median():.1f}')
axes[0].set_xlabel(col)
axes[0].set_ylabel('Frecuencia')
axes[0].xaxis.set_major_locator(plt.MultipleLocator(1))
total = df[col].dropna().shape[0]
for patch, freq in zip(patches, n):
    if freq > 0:
        axes[0].text(patch.get_x() + patch.get_width() / 2, freq + total * 0.005,
                     f'{int(freq):,}', ha='center', va='bottom', fontsize=10)

data_by_class = [df[df['AdoptionSpeed'] == cls][col].dropna() for cls in range(5)]
bp = axes[1].boxplot(data_by_class, patch_artist=True, medianprops=dict(color='black', linewidth=2), showfliers=True)
for patch, color in zip(bp['boxes'], PALETTE_GRAD):
    patch.set_facecolor(color)
axes[1].set_xticklabels(['0', '1', '2', '3', '4'])
axes[1].set_xlabel('AdoptionSpeed')
axes[1].set_title(f'{col} por clase de AdoptionSpeed')
axes[1].yaxis.set_major_locator(plt.MultipleLocator(1))

plt.suptitle(f'Variable: {col}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


Conclusión - Cantidad de mascotas por registro:

* La mayoría de los registros corresponden a publicaciones con 1 mascota. Hay algunos casos de 2 y 3, llegando a casos extremos de hasta 20 mascotas en una sola publicación (típicamente camadas).
* La clase 4 (no adoptado) es la única que tiene una caja visible en el boxplot, con mediana en 1 y Q3 en 2, lo que indica que en los avisos que nunca fueron adoptados hay una mayor proporción de publicaciones con más de una mascota. 
* Publicar varias mascotas juntas podría dificultar la adopción, posiblemente porque los adoptantes prefieren llevarse una sola mascota a la vez. Esto es consistente con lo observado anteriormente en la variable Género, donde los grupos mixtos (que típicamente corresponden a camadas) presentaban mayor proporción de no adoptados.

Se procede a armar un gráfico que muestre la velocidad de adopción según se trate de una mascota individual, un grupo pequeño (2-3 mascotas) o un grupo grande (4 o más mascotas):

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Categorización en 3 grupos
def categorizar_grupo(q):
    if q == 1:
        return 'Individual'
    elif q <= 3:
        return 'Grupo pequeño (2-3)'
    else:
        return 'Grupo grande (4+)'

df['TipoGrupo'] = df['Quantity'].apply(categorizar_grupo)

orden = ['Individual', 'Grupo pequeño (2-3)', 'Grupo grande (4+)']

cross_qty = df.groupby(['TipoGrupo', 'AdoptionSpeed']).size().unstack(fill_value=0)
cross_qty = cross_qty.reindex(orden)
cross_qty_pct = cross_qty.div(cross_qty.sum(axis=1), axis=0) * 100

cross_qty_pct.plot(kind='bar', ax=ax, color=PALETTE_GRAD, width=0.6)
ax.set_title('AdoptionSpeed: Individual vs Grupos')
ax.set_xlabel('')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Clase', labels=list(adoption_map.values()), fontsize=8)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=9, padding=2)

freq_total = cross_qty.sum(axis=1)
ax2 = ax.twinx()
ax2.plot(range(len(freq_total)), freq_total.values,
         color='black', marker='o', linewidth=1.5, markersize=4, linestyle='--')
ax2.set_ylabel('Frecuencia total', fontsize=8)
ax2.tick_params(axis='y', labelsize=9)
ax2.set_ylim(0, freq_total.values.max() * 1.3)
ax2.yaxis.grid(False)

plt.tight_layout()
plt.show()

Conclusión:

Se observa que la proporción de clase 0 y 1 (adoptados entre el primer día y la primera semana) es similar para mascotas individuales o grupos pequeños, y menor para grupos grandes.
La mayor diferencia se observa en los No Adoptados. La proporción es menor en mascotas individuales, y crece a medida que se agranda el grupo de mascotas.

### Fee (Tarifa de adopción)

In [ ]:
# Fee
col = 'Fee'
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

n, bins, patches = axes[0].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', align='mid')
axes[0].set_title(f'{col} — Distribución\nMedia: {df[col].mean():.1f} | Mediana: {df[col].median():.1f}')
axes[0].set_xlabel(col)
axes[0].set_ylabel('Frecuencia')
total = df[col].dropna().shape[0]
for patch, freq in zip(patches, n):
    if freq > 0:
        axes[0].text(patch.get_x() + patch.get_width() / 2, freq + total * 0.005,
                     f'{int(freq):,}', ha='center', va='bottom', fontsize=10)

data_by_class = [df[df['AdoptionSpeed'] == cls][col].dropna() for cls in range(5)]
bp = axes[1].boxplot(data_by_class, patch_artist=True, medianprops=dict(color='black', linewidth=2), showfliers=True)
for patch, color in zip(bp['boxes'], PALETTE_GRAD):
    patch.set_facecolor(color)
axes[1].set_xticklabels(['0', '1', '2', '3', '4'])
axes[1].set_xlabel('AdoptionSpeed')
axes[1].set_title(f'{col} por clase de AdoptionSpeed')

plt.suptitle(f'Variable: {col}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


Conclusión - Tarifa de adopción:

* La gran mayoría de los registros tienen Fee = 0, lo que implica que la mayoría de los rescatistas/refugios ofrecen adopciones gratuitas, posiblemente como estrategia para favorecer la colocación de animales rápidamente.
* El boxplot no permite identificar diferencias claras entre clases, dado que la fuerte concentración de valores en 0 aplasta la caja y solo deja visibles los outliers. 

Se decide agregar una visualización que distinga entre "Gratis" y "Con costo" para evidenciar si hay alguna relación con la velocidad de adopción:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

df['HasFee'] = (df['Fee'] > 0).map({True: 'Con costo', False: 'Gratuito'})
cross_fee = df.groupby(['HasFee', 'AdoptionSpeed']).size().unstack(fill_value=0)
cross_fee_pct = cross_fee.div(cross_fee.sum(axis=1), axis=0) * 100
cross_fee_pct.plot(kind='bar', ax=ax, color=PALETTE_GRAD, width=0.6)
ax.set_title('AdoptionSpeed según costo')
ax.set_xlabel('')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Clase', labels=list(adoption_map.values()), fontsize=7)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=9, padding=2)

freq_total = cross_fee.sum(axis=1)
ax2 = ax.twinx()
ax2.plot(range(len(freq_total)), freq_total.values,
         color='black', marker='o', linewidth=1.5, markersize=4, linestyle='--')
ax2.set_ylabel('Frecuencia total', fontsize=8)
ax2.tick_params(axis='y', labelsize=7)
ax2.set_ylim(0, freq_total.values.max() * 1.3)
ax2.yaxis.grid(False)

plt.tight_layout()
plt.show()

Conclusión:

Ambas categorías tienen comportamiento similar, aunque "Con costo" agrupa un mayor porcentaje de mascotas no adoptadas. Esto puede deberse a que la tarifa actúa como barrera para algunos adoptantes, o a que las mascotas con costo correspondan a perfiles más específicos (razas particulares, animales con tratamientos veterinarios incluidos, etc.).

### PhotoAmt ("Photo Amount" o cantidad de fotos por publicación) y VideoAmt ("Video Amount" o cantidad de videos por publicación)

In [ ]:
# PhotoAmt 
col = 'PhotoAmt'
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

n, bins, patches = axes[0].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', align='mid')
axes[0].set_title(f'{col} — Distribución\nMedia: {df[col].mean():.1f} | Mediana: {df[col].median():.1f}')
axes[0].set_xlabel(col)
axes[0].set_ylabel('Frecuencia')
total = df[col].dropna().shape[0]
for patch, freq in zip(patches, n):
    if freq > 0:
        axes[0].text(patch.get_x() + patch.get_width() / 2, freq + total * 0.005,
                     f'{int(freq):,}', ha='center', va='bottom', fontsize=10)

data_by_class = [df[df['AdoptionSpeed'] == cls][col].dropna() for cls in range(5)]
bp = axes[1].boxplot(data_by_class, patch_artist=True, medianprops=dict(color='black', linewidth=2), showfliers=True)
for patch, color in zip(bp['boxes'], PALETTE_GRAD):
    patch.set_facecolor(color)
axes[1].set_xticklabels(['0', '1', '2', '3', '4'])
axes[1].set_xlabel('AdoptionSpeed')
axes[1].set_title(f'{col} por clase de AdoptionSpeed')
axes[1].yaxis.set_major_locator(plt.MultipleLocator(2))

plt.suptitle(f'Variable: {col}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# VideoAmt
col = 'VideoAmt'
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

n, bins, patches = axes[0].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', align='mid')
axes[0].set_title(f'{col} — Distribución\nMedia: {df[col].mean():.1f} | Mediana: {df[col].median():.1f}')
axes[0].set_xlabel(col)
axes[0].set_ylabel('Frecuencia')
total = df[col].dropna().shape[0]
for patch, freq in zip(patches, n):
    if freq > 0:
        axes[0].text(patch.get_x() + patch.get_width() / 2, freq + total * 0.005,
                     f'{int(freq):,}', ha='center', va='bottom', fontsize=10)

data_by_class = [df[df['AdoptionSpeed'] == cls][col].dropna() for cls in range(5)]
bp = axes[1].boxplot(data_by_class, patch_artist=True, medianprops=dict(color='black', linewidth=2), showfliers=True)
for patch, color in zip(bp['boxes'], PALETTE_GRAD):
    patch.set_facecolor(color)
axes[1].set_xticklabels(['0', '1', '2', '3', '4'])
axes[1].set_xlabel('AdoptionSpeed')
axes[1].set_title(f'{col} por clase de AdoptionSpeed')

plt.suptitle(f'Variable: {col}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusiones:

Photo Amount:
* Del histograma se observa que la mayoría de las mascotas (12.132) tienen entre 1 y 5 fotos. Solo 341 no tienen foto, y las demás tienen más de 5, llegando a un máximo de 30.
* En el boxplot se observa que la clase 4 (no adoptado) tiene una mediana de fotos notablemente más baja que el resto de las clases, e incluye más casos con 0 fotos. Esto sugiere que tener al menos una foto es un factor relevante para que la mascota llegue a ser adoptada, más que la cantidad exacta de fotos. La diferencia entre las clases 0 a 3 es menos marcada, lo que refuerza la idea de que el umbral importante es "con foto vs. sin foto".


Video Amount: 
* La gran mayoría de los registros no tiene video (VideoAmt = 0). Esta concentración extrema deja al boxplot prácticamente sin información: como casi todos los valores son 0 en todas las clases, no se puede evidenciar una relación clara entre la cantidad de videos y la velocidad de adopción. 

Para aportar más información en esta etapa exploratoria, se agregan visualizaciones que distinguen entre registros Con/Sin Foto y Con/Sin Video para analizar su comportamiento con respecto a la velocidad de adopción.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Con / Sin foto 
df['HasPhoto'] = (df['PhotoAmt'].fillna(0) > 0).map({True: 'Con foto', False: 'Sin foto'})
cross_photo = df.groupby(['HasPhoto', 'AdoptionSpeed']).size().unstack(fill_value=0)
cross_photo_pct = cross_photo.div(cross_photo.sum(axis=1), axis=0) * 100
cross_photo_pct.plot(kind='bar', ax=axes[0], color=PALETTE_GRAD, width=0.6)
axes[0].set_title('AdoptionSpeed según fotos')
axes[0].set_xlabel('')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Clase', labels=list(adoption_map.values()), fontsize=7)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=9, padding=2)

freq_photo = cross_photo.sum(axis=1)
ax2 = axes[0].twinx()
ax2.plot(range(len(freq_photo)), freq_photo.values,
         color='black', marker='o', linewidth=1.5, markersize=4, linestyle='--')
ax2.set_ylabel('Frecuencia total', fontsize=8)
ax2.tick_params(axis='y', labelsize=9)
ax2.set_ylim(0, freq_photo.values.max() * 1.3)
ax2.yaxis.grid(False)

# Con / Sin video 
df['HasVideo'] = (df['VideoAmt'].fillna(0) > 0).map({True: 'Con video', False: 'Sin video'})
cross_video = df.groupby(['HasVideo', 'AdoptionSpeed']).size().unstack(fill_value=0)
cross_video_pct = cross_video.div(cross_video.sum(axis=1), axis=0) * 100
cross_video_pct.plot(kind='bar', ax=axes[1], color=PALETTE_GRAD, width=0.6, legend=False)
axes[1].set_title('AdoptionSpeed según video')
axes[1].set_xlabel('')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=0)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=9, padding=2)

freq_video = cross_video.sum(axis=1)
ax3 = axes[1].twinx()
ax3.plot(range(len(freq_video)), freq_video.values,
         color='black', marker='o', linewidth=1.5, markersize=4, linestyle='--')
ax3.set_ylabel('Frecuencia total', fontsize=8)
ax3.tick_params(axis='y', labelsize=9)
ax3.set_ylim(0, freq_video.values.max() * 1.3)
ax3.yaxis.grid(False)

plt.suptitle('Fotos y videos vs. AdoptionSpeed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusiones:

* Se observa que las mascotas sin foto tienen una alta proporción de no adoptados, lo que confirmaría que el tener al menos una foto es un factor relevante para la adopción. 
* Las mascotas sin video también tienen mayor proporción de no adoptados, aunque en menor medida; diferencia que es además menos robusta dado que la enorme mayoría de los registros no tiene video.

## 3.11 Correlación entre Variables Numéricas

Se analiza la correlación entre variables con el coeficiente de Spearman, ya que admite variables discretas (como Quantity o PhotoAmt/VideoAmt) y no requiere probar distribución normal.

In [ ]:
corr_cols = ['Age', 'Quantity', 'Fee', 'PhotoAmt', 'VideoAmt']

corr_matrix = df[corr_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de correlación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusión:

De la matriz se observa que las correlaciones son todas bajas (ninguna supera 0.25 en valor absoluto), lo que indica que no hay multicolinealidad importante entre las variables numéricas, lo cual es favorable para el modelado.

## 3.12 Asociación entre Variables Categóricas

Para visualizar la asociación entre variables categóricas de a pares se construye un gráfico de V de Cramér. Está basada en el test de Chi-cuadrado. El valor va de 0 (sin asociación) a 1 (asociación perfecta), y no tiene valores negativos (a diferencia de Pearson o Spearman, que van de -1 a 1) porque mide la fuerza de la asociación pero no su dirección.
Se cumplen los supuestos para usar V de Cramér:

1. Variables categóricas.
2. Frecuencias esperadas suficientes (mayores o iguales a 5).
3. Observaciones independientes.

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

cat_cols = ['Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3',
            'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized',
            'Health', 'State']

# Calcular matriz de Cramér's V
cramers_matrix = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)

for col1 in cat_cols:
    for col2 in cat_cols:
        if col1 == col2:
            cramers_matrix.loc[col1, col2] = 1.0
        else:
            sub = df[[col1, col2]].dropna()
            cramers_matrix.loc[col1, col2] = cramers_v(sub[col1], sub[col2])

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(cramers_matrix, dtype=bool))
sns.heatmap(cramers_matrix.astype(float), mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', vmin=0, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title("Matriz de asociación — V de Cramér (variables categóricas)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusiones: 
* Se observa poca relación entre la mayoría de las variables. Las que presentan una relación fuerte son:

    * Type y Breed1 = 0.99. No provee de gran información, dado que cada raza pertenece exclusivamente a una especie (perro o gato), por lo que conocer Breed1 determina prácticamente el valor de Type.
    * Dewormed y Vaccinated = 0.75. Ambos son tratamientos veterinarios pre-adopción, y suelen aplicarse en conjunto cuando la mascota pasa por el protocolo del refugio. Un razonamiento similar explica las correlaciones moderadas entre Sterilized y Vaccinated (= 0.49), y Sterilized y Dewormed (= 0.45). Aunque la correlación es alta, no es "perfecta" (hay casos donde una variable difiere de la otra: mascotas vacunadas pero no desparasitadas, o viceversa)), por lo que conviene mantenerlas como variables separadas.
    * FurLength y Breed1 = 0.54. Tampoco aporta gran información, ya que cada raza tiene un tipo de pelaje característico.



## 3.13 Perfil del Rescatista

In [ ]:
# Stats básicas por rescatista 
rescuer_stats = df.groupby('RescuerID').agg(
    n_publicaciones=('PetID', 'count'),
    adoption_speed_media=('AdoptionSpeed', 'mean'),
    pct_adoptados=('AdoptionSpeed', lambda x: (x < 4).mean() * 100),
    pct_rapidos=('AdoptionSpeed', lambda x: (x <= 1).mean() * 100),
).reset_index()

print(f'Total rescatistas:     {len(rescuer_stats):,}')
print(f'Mínimo publicaciones:  {rescuer_stats["n_publicaciones"].min()}')
print(f'Máximo publicaciones:  {rescuer_stats["n_publicaciones"].max()}')
print(f'Media publicaciones:   {rescuer_stats["n_publicaciones"].mean():.1f}')
print(f'Mediana publicaciones: {rescuer_stats["n_publicaciones"].median():.0f}')

# Perfiles
rescuer_stats['Perfil'] = pd.cut(
    rescuer_stats['n_publicaciones'],
    bins=[0, 1, 5, 20, rescuer_stats['n_publicaciones'].max()],
    labels=['Ocasional (1)', 'Pequeño (2-5)', 'Mediano (6-20)', 'Alto volumen (>20)']
)

PERFIL_ORDER  = ['Ocasional (1)', 'Pequeño (2-5)', 'Mediano (6-20)', 'Alto volumen (>20)']
PERFIL_COLORS = ['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']

# Gráfico 1: Histograma de publicaciones
fig, ax = plt.subplots(figsize=(10, 5))

n, bins, patches = ax.hist(rescuer_stats['n_publicaciones'], bins=40,
                            color='steelblue', edgecolor='white', align='mid')
ax.set_title('Distribución de publicaciones por rescatista', fontsize=13, fontweight='bold')
ax.set_xlabel('Cantidad de publicaciones')
ax.set_ylabel('Frecuencia')
ax.xaxis.set_major_locator(plt.MultipleLocator(20))

total = len(rescuer_stats)
for patch, freq in zip(patches, n):
    if freq > 0:
        ax.text(patch.get_x() + patch.get_width() / 2, freq + total * 0.003,
                f'{int(freq):,}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

# Gráfico 2: Cantidad de rescatistas por perfil
perfil_counts = rescuer_stats['Perfil'].value_counts().reindex(PERFIL_ORDER)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(perfil_counts.index, perfil_counts.values,
              color=PERFIL_COLORS, edgecolor='white')
ax.set_title('Cantidad de rescatistas por perfil', fontsize=13, fontweight='bold')
ax.set_ylabel('Cantidad de rescatistas')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)

for bar, v in zip(bars, perfil_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{v:,}\n({v/len(rescuer_stats)*100:.1f}%)',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Gráfico 3: AdoptionSpeed por perfil + línea de frecuencia
# Asegurarse de que Perfil esté en df
if 'Perfil' not in df.columns:
    df = df.merge(rescuer_stats[['RescuerID', 'Perfil']], on='RescuerID', how='left')


# Unir perfil al df original
fig, ax = plt.subplots(figsize=(12, 5))

cross_perfil = df.groupby(['Perfil', 'AdoptionSpeed']).size().unstack(fill_value=0)
cross_perfil = cross_perfil.reindex(PERFIL_ORDER)
cross_perfil_pct = cross_perfil.div(cross_perfil.sum(axis=1), axis=0) * 100

cross_perfil_pct.plot(kind='bar', ax=ax, color=PALETTE_GRAD, width=0.7)
ax.set_title('AdoptionSpeed por perfil de rescatista', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='AdoptionSpeed', labels=list(adoption_map.values()), fontsize=7)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=9, padding=2)

plt.tight_layout()
plt.show()

Conclusiones: 

* Gráfico 1: Distribución de publicaciones por rescatista

Se observa una distribución fuertemente asimétrica positiva: la gran mayoría de los rescatistas tiene menos de 20 publicaciones (la masa se concentra en los valores más bajos), mientras que unos pocos casos extremos presentan más de 100 e incluso más de 400. Esto refleja que la actividad de rescate en la plataforma no se distribuye uniformemente, sino que hay rescatistas mucho más activos que otros.


* Gráfico 2: Cantidad de rescatistas por perfil

La gran mayoría de los rescatistas son Ocasionales (1 sola publicación), seguidos por los de Pequeño volumen (2-5 publicaciones). Los perfiles Mediano (6-20) y Alto volumen (>20) representan una minoría dentro del padrón de rescatistas. Es importante tener presente esta distribución al interpretar el siguiente gráfico, ya que la masa central del dataset está concentrada en los perfiles con menos publicaciones por persona.


* Gráfico 3: AdoptionSpeed por perfil de rescatista

Se observa una relación clara: a mayor volumen de publicaciones del rescatista, menor proporción de mascotas no adoptadas. Los ocasionales presentan el mayor porcentaje de no adoptados, seguidos por Pequeño volumen, mientras que los de Alto volumen muestran la menor cantidad de casos no adoptados.
Por otro lado, los rescatistas de Alto volumen muestran menos adopciones rápidas (clases 0 y 1) y más adopciones en clases intermedias (2 y 3). Esto sugiere que los rescatistas experimentados son más efectivos en concretar adopciones, posiblemente por su experiencia, red de contactos y persistencia, aunque sus adopciones tardan más en promedio (quizá por manejar más casos a la vez o atender mascotas más difíciles de adoptar).
Cabe aclarar que cada rescatista Ocasional aporta una única observación, lo que hace que su distribución sea más variable. Aun así, dado que constituyen la gran mayoría (Gráfico 2), su patrón es relevante para el comportamiento global de la plataforma.

Se destaca que hay 5 rescatistas con más de 150 publicaciones cada uno. Resulta de interés saber cómo se comportan estos casos. Se procede a identificarlos y obtener los siguientes datos:

* cantidad de avisos que publicó ese rescatista
* adoption_speed_media: promedio de AdoptionSpeed de todas sus mascotas
* pct_adoptados: porcentaje de sus mascotas que sí fueron adoptadas (AdoptionSpeed < 4)
* pct_rapidos: porcentaje de sus mascotas adoptadas el mismo día o en la primera semana (AdoptionSpeed ≤ 1)

In [ ]:
# Identificación rescatistas con más de 150 publicaciones
top_rescuers = rescuer_stats[rescuer_stats['n_publicaciones'] > 150].sort_values('n_publicaciones', ascending=False)
print('Rescatistas con más de 150 publicaciones:')
print(top_rescuers[['RescuerID', 'n_publicaciones', 'adoption_speed_media', 'pct_adoptados', 'pct_rapidos']].to_string(index=False))

# Distribución de AdoptionSpeed para cada uno
fig, axes = plt.subplots(1, len(top_rescuers), figsize=(4 * len(top_rescuers), 5))

for ax, (_, row) in zip(axes, top_rescuers.iterrows()):
    sub = df[df['RescuerID'] == row['RescuerID']]
    cross = sub['AdoptionSpeed'].value_counts().reindex(range(5), fill_value=0)
    cross_pct = cross / cross.sum() * 100

    bars = ax.bar([adoption_map[i] for i in range(5)], cross_pct.values,
                  color=PALETTE_GRAD, edgecolor='white')
    ax.set_title(f'ID: {row["RescuerID"][:8]}...\n{int(row["n_publicaciones"])} publicaciones',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylim(0, 65)

    for bar, pct in zip(bars, cross_pct.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{pct:.0f}%', ha='center', va='bottom', fontsize=11)

plt.suptitle('Distribución de AdoptionSpeed — Rescatistas con >150 publicaciones',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Conclusiones:

Al tener tantas publicaciones, estos 5 rescatistas podrían presentar un patrón propio (relacionado con su experiencia, habilidad o compromiso) para lograr que las mascotas se adopten, manifestando una efectividad característica como la que muestran los gráficos de barras de cada uno.
Los 5 rescatistas top no se comportan de manera homogénea: hay diferencias muy marcadas entre ellos. Por ejemplo:

* El rescatista con 459 publicaciones presenta una alta tasa de adopción: solo el 2% de sus mascotas no fue adoptado, aunque muchas se adoptaron en períodos largos (entre el 2do y 3er mes).
* En contraste, el rescatista con 156 publicaciones presenta una efectividad notablemente menor, con un 58% de sus mascotas sin adoptar.

Esta heterogeneidad muestra que el volumen de publicaciones por sí solo no determina la efectividad: la identidad específica del rescatista influye fuertemente. Si estos mismos rescatistas aparecen en futuras publicaciones, sus patrones de efectividad podrían mantenerse, por lo que la identidad de los rescatistas con mayor volumen es una variable potencialmente útil para el modelo (a explorar en el feature engineering).

## 3.14 Descripción textual

Habíamos detectado que solo 13 registros no tenían descripción. Por lo que no sería representativo analizar si tener o no descripción afecta en la velocidad de adopción.

Sí podemos analizar si la longitud de la descripción guarda relación o no a la velocidad de adopción.

In [ ]:
df['DescLen'] = df['Description'].fillna('').apply(len)

fig, ax = plt.subplots(figsize=(8, 5))

data_desc = [df[df['AdoptionSpeed']==cls]['DescLen'] for cls in range(5)]
bp = ax.boxplot(data_desc, patch_artist=True,
                medianprops=dict(color='black', linewidth=2),
                showfliers=False)
for patch, color in zip(bp['boxes'], PALETTE_GRAD):
    patch.set_facecolor(color)
ax.set_xticklabels(['0','1','2','3','4'])
ax.set_title('Longitud de descripción por clase')
ax.set_xlabel('AdoptionSpeed')
ax.set_ylabel('Caracteres')
ax.set_ylim(-100, 1000)

plt.tight_layout()
plt.show()

Comentario sobre el gráfico: Para una mejor de la visualización de las tendencias centrales, el gráfico fue acotado excluyendo los outliers.

Conclusión:

Se observa que las distribuciones de longitud de descripción son similares para las cinco clases de AdoptionSpeed, sin una tendencia clara que asocie la longitud con la velocidad de adopción.
Es posible que la información relevante no esté en cuánto se escribe sino en qué se escribe: la calidad del contenido, el tono, las palabras clave usadas o incluso el idioma en que está escrita la descripción. Esto se podrá explorar más adelante con herramientas de procesamiento de texto (NLP).

## 3.15 Resumen ejecutivo

In [ ]:
print(df.columns.tolist())

In [ ]:
separador = '=' * 65

print(separador)
print('   RESUMEN EDA — PetFinder Adoption Prediction')
print(separador)


cols_originales = ['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 
                   'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 
                   'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 
                   'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed']


# Dataset
print(f'''
📦 DATASET
   Registros  : {df.shape[0]:,}
   Variables  : {len(cols_originales)} originales
   Duplicados : {df.duplicated().sum()}
   Perros     : {(df["Type"]==1).sum():,} ({(df["Type"]==1).mean()*100:.1f}%)
   Gatos      : {(df["Type"]==2).sum():,} ({(df["Type"]==2).mean()*100:.1f}%)
''')


# Variable objetivo
print('📊 VARIABLE OBJETIVO — AdoptionSpeed')
for cls, lbl in adoption_map.items():
    n   = (df['AdoptionSpeed'] == cls).sum()
    pct = n / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'   {lbl:<15} {n:>5,} ({pct:>5.1f}%) {bar}')

adoptados_pct = (df['AdoptionSpeed'] < 4).mean() * 100
print(f'\n   ✅ Tasa global de adopción: {adoptados_pct:.1f}%')

# Calidad de datos
sin_nombre = (df['Name'].isna() | (df['Name'].str.strip() == 'No Name')).sum()
print(f'''🔍 CALIDAD DE DATOS
   Faltantes relevantes:
   - Name        : {sin_nombre:,} ({sin_nombre/len(df)*100:.1f}%) — nulos reales + "No Name"
   - Description : {df["Description"].isna().sum()} (0.09%) — no relevante
   Sin duplicados en el dataset ✓
''')

# Variables numéricas
print('📈 VARIABLES NUMÉRICAS (medianas)')
num_cols = ['Age', 'Quantity', 'Fee', 'PhotoAmt', 'VideoAmt']
for col in num_cols:
    print(f'   {col:<12}: media={df[col].mean():.1f} | mediana={df[col].median():.1f} | máx={df[col].max():.0f}')

sin_foto = (df['PhotoAmt'].fillna(0) == 0).sum()
print(f'\n   📷 Sin foto    : {sin_foto:,} ({sin_foto/len(df)*100:.1f}%)')
sin_video = (df['VideoAmt'].fillna(0) == 0).sum()
print(f'   🎥 Sin video   : {sin_video:,} ({sin_video/len(df)*100:.1f}%)')
gratuitos = (df['Fee'] == 0).sum()
print(f'   💰 Gratuitos   : {gratuitos:,} ({gratuitos/len(df)*100:.1f}%)\n')

# Razas
razas_genericas = ['Mixed Breed', 'Domestic Short Hair', 'Domestic Medium Hair', 'Domestic Long Hair']
raza_mixta = ((df['Breed1_label'].notna() & df['Breed2_label'].notna()) | df['Breed1_label'].isin(razas_genericas)).sum()
raza_pura  = (df['Breed1_label'].notna() & df['Breed2_label'].isna() & ~df['Breed1_label'].isin(razas_genericas)).sum()
print(f'''🐾 RAZAS
   Razas únicas Breed1 : {df["Breed1_label"].nunique()}
   Razas únicas Breed2 : {df["Breed2_label"].nunique()}
   Raza pura           : {raza_pura:,} ({raza_pura/len(df)*100:.1f}%)
   Raza mixta/genérica : {raza_mixta:,} ({raza_mixta/len(df)*100:.1f}%)
''')

# Rescatistas
rescuer_counts = df['RescuerID'].value_counts()
print(f'''👤 RESCATISTAS
   Total rescatistas      : {df["RescuerID"].nunique():,}
   Publicaciones (mediana): {rescuer_counts.median():.0f}
   Publicaciones (máximo) : {rescuer_counts.max()}
   Con >20 publicaciones  : {(rescuer_counts > 20).sum():,} ({(rescuer_counts > 20).mean()*100:.1f}%)
   Con >150 publicaciones : {(rescuer_counts > 150).sum()}
''')

# Hallazgos clave
print(separador)
print('🔑 HALLAZGOS CLAVE')
print(separador)
hallazgos = [
    ('Variable objetivo',  'Clases relativamente balanceadas. Clase 0 (1er día) la menos frecuente (2.7%) y Clase 2 (1er mes) la más frecuente (26.9%)'),
    ('Edad',               'Las mascotas más jóvenes se adoptan más rápido.'),
    ('Fotos',              'Las mascotas con fotos tienen mayor velocidad de adopción'),
    ('Costo',              'Las mascotas gratuitas se adoptan más rápido'),
    ('Salud',              'Variable con mayor impacto: lesión grave → ~45% no adoptados'),
    ('Cantidad',           'Avisos con >1 mascota tienen mayor tasa de no adopción (clase 4)'),
    ('Raza',               'Los perros raza pura se adoptan más rápido que los raza mixta. Para gatos sin diferencias marcadas entre raza pura y mixta en velocidad de adopción'),
    ('Estado',             '84% de las mascotas concentradas en 2 Estados'),
    ('Correlación',        'Correlaciones numéricas bajas (Spearman). Sin multicolinealidad relevante'),
    ('Asociación categ.',  'V de Cramér: State y Breed1 muestran mayor asociación entre categóricas'),
]
for tema, conclusion in hallazgos:
    print(f'\n   📌 {tema}')
    print(f'      {conclusion}')

print(f'\n{separador}')
print('🛠️  FEATURES CANDIDATAS PARA FEATURE ENGINEERING')
print(separador)
features = [
    'HasName         → si la mascota tiene nombre real',
    'HasPhoto        → si tiene al menos 1 foto',
    'HasFee          → si tiene costo de adopción',
    'TipoRaza        → Raza pura vs. mixta',
    'n_colores       → cantidad de colores únicos',
    'PerfilRescatista → volumen de publicaciones del rescatador',
    'DescLen         → longitud de la descripción del aviso',
    'IsGroup         → si el aviso incluye más de 1 mascota',
]
for f in features:
    print(f'   ✦ {f}')

print(f'\n{separador}')


## 3.16 Dashboard interactivo con Streamlit

Las celdas de esta sección hacen todo desde el notebook, sin salir de VS Code:
1. **Celda 17a** — instala dependencias si faltan
2. **Celda 17b** — genera `dashboard.py` en la carpeta del notebook
3. **Celda 17c** — lanza Streamlit en segundo plano (se abre en `http://localhost:8501`)
4. **Celda 17d** — detiene el servidor cuando terminás

In [ ]:
# 17a. Verificar / instalar dependencias
import subprocess, sys

for pkg in ['streamlit', 'plotly']:
    try:
        __import__(pkg)
        print(f' {pkg} — OK')
    except ImportError:
        print(f' Instalando {pkg} para asegurar reproducibilidad...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f' {pkg} — instalado')

In [ ]:
# 17c. Lanzar Streamlit en segundo plano
import subprocess, sys, os, time
from pathlib import Path

# Usamos la ruta absoluta basada en tu configuración de carpetas
dashboard_path = BASE_DIR / "EDAdashboard.py"
puerto = "8501"

if not dashboard_path.exists():
    print(f" Error: No se encuentra el archivo {dashboard_path}")
else:
    # Lanzar como proceso independiente usando el ejecutable de Python actual
    _streamlit_proc = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', str(dashboard_path),
         '--server.headless', 'true', # true es mejor para servidores automáticos
         '--server.port', puerto],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(5)  # Damos tiempo suficiente para que el servidor levante
    print(f' Dashboard lanzado exitosamente.')
    print(f' Accedé aquí: http://localhost:{puerto}')
    print(f' PID del proceso: {_streamlit_proc.pid}')
    print('  Nota: Ejecutá la celda 17d para liberar el puerto al terminar.')

In [ ]:
# 17d. Detener el servidor Streamlit
try:
    if '_streamlit_proc' in globals():
        _streamlit_proc.terminate()
        _streamlit_proc.wait() # Espera a que el proceso se limpie
        print(' Servidor Streamlit detenido correctamente.')
        del _streamlit_proc
    else:
        print('ℹ No hay servidor activo detectado.')
except Exception as e:
    print(f' Error al intentar detener el servidor: {e}')

---
# Sección 4. Feature Engineering

In [ ]:
# Creamos df2 solo con las variables originales de df y sus labels
cols = ['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed', 'Type_label', 'Gender_label', 'MaturitySize_label', 'FurLength_label', 'Health_label', 'Vaccinated_label', 'Dewormed_label', 'Sterilized_label', 'Breed1_label', 'Breed2_label', 'Color1_label', 'Color2_label', 'Color3_label', 'State_label', 'AdoptionSpeed_label']

df2 = df[cols].copy()

print(df2.shape)

label_cols = [col for col in df2.columns if col.endswith('_label')]
original_cols = [col for col in df2.columns if not col.endswith('_label')]

print(f'Variables originales: {len(original_cols)}')
print(f'Variables _label:     {len(label_cols)}')

## 4.1 Split


Se separan los datos antes de la creación de variables para prevenir la filtración de información (data leakage), asegurando que el modelo se entrene y valide exclusivamente con información disponible en el origen, sin que el conocimiento del conjunto de test influya en el diseño de las features.

In [ ]:
#Separo un 20% para test estratificado por target
train, test = train_test_split(df2,
                               test_size = TEST_SIZE,
                               random_state = SEED,
                               stratify = df2.AdoptionSpeed)

print(f"Train: {len(train):,} filas")
print(f"Test : {len(test):,} filas")

print(train.shape)
print(test.shape)


## 4.2. Feature Engineering

#### "tiene_nombre"

Clasifica a las mascotas determinando si tienen nombre real y propio (valor 1), descartando mediante expresiones regulares (Regex) a las que no tienen nombre, usan identificadores numéricos, o figuran con términos genéricos o contextuales como "cachorro", "adoptar" o "desconocido" (valor 0).

In [ ]:
import re

sin_nombre_terms = [
    r'no name', r'unnamed', r'unknown', r'nameless', r'not name', 
    r'not yet', r'\bnone\b', r'\btba\b', r'\btbd\b', r'\(no name\)', r'sin_nombre',
    r'sin nombre', r'\bnil\b', r'\bna\b', r'\bn/a\b'
]
sin_nombre_pattern = re.compile('|'.join(sin_nombre_terms))

context_terms = [
    r'\badopt\b', r'\bstray\b', r'\bfound\b', r'\brescue\b', r'\burgent\b', r'\bneed\b', 
    r'\bfoster\b', r'\bfree\b', r'\bmonth\b', r'\bmonths\b', r'\bweek\b', r'\bweeks\b', 
    r'\byear\b', r'\byears\b', r'\bold\b'
]
context_pattern = re.compile('|'.join(context_terms))

genericos = [
    r'\bpuppy\b', r'\bpuppies\b', r'\bpup\b', r'\bpups\b',
    r'\bkitten\b', r'\bkittens\b', r'\bkitty\b', r'\bkitties\b',
    r'\bcat\b', r'\bcats\b', r'\bdog\b', r'\bdogs\b', r'\bdoggie\b', r'\bdoggy\b', r'\bdoggies\b',
    r'\bboy\b', r'\bgirl\b', r'\bmale\b', r'\bfemale\b',
    r'\bmom\b', r'\bmama\b', r'\bmommy\b', r'\bmother\b', r'\bmomma\b',
    r'\bbaby\b', r'\bbabies\b', r'\bsiblings\b', r'\btwins\b',
    r'\bbrother\b', r'\bsister\b', r'\bmix\b', r'\bbreed\b',
    r'\bpet\b', r'\bpets\b'
]
generico_pattern = re.compile('|'.join(genericos))


def clasificar_nombre(nombre):
    if not isinstance(nombre, str):
        return 0
    
    n = nombre.lower().strip()
    
    if len(n) <= 1 or n.isnumeric():
        return 0
        
    if re.match(r'^[a-z]\d{1,2}$', n):
        return 0
    
    if sin_nombre_pattern.search(n):
        return 0
    
    if context_pattern.search(n):
        return 0
    
    if generico_pattern.search(n):
        return 0
        
    return 1


train['tiene_nombre'] = train['Name'].apply(clasificar_nombre)
test['tiene_nombre'] = test['Name'].apply(clasificar_nombre)

#### "largo_descripcion"

Calcula la cantidad de palabras en la descripcion y le aplica un logaritmo para darle mas peso a las diferencias de descripciones cortas y mas suavizado en las largas. 
NO se toca nada que tenga que ver con idiomas.

In [ ]:
def calcular_largo_descripcion(df):
    descripciones = df['Description'].fillna('')
    conteo_palabras = descripciones.apply(lambda x: len(str(x).split()))
    df['largo_descripcion'] = np.log1p(conteo_palabras)
    
    return df

# Aplicación
train = calcular_largo_descripcion(train)
test = calcular_largo_descripcion(test)


#### "bins_edad"

El objetivo es discretizar la edad en rangos:
- 0: desconocido
- 1: 0–2 meses
- 2: 3–6 meses
- 3: 7–12 meses
- 4: 12 - 24 meses
- 5: mas de 36 meses

In [ ]:
def crear_bins_edad(df):
    df['bins_edad'] = 0  # default (por si hay errores)
    
    df.loc[df['Age'] == 0, 'bins_edad'] = 0
    df.loc[(df['Age'] >= 1) & (df['Age'] <= 2), 'bins_edad'] = 1
    df.loc[(df['Age'] >= 3) & (df['Age'] <= 6), 'bins_edad'] = 2
    df.loc[(df['Age'] >= 7) & (df['Age'] <= 11), 'bins_edad'] = 3
    df.loc[(df['Age'] >= 12) & (df['Age'] <= 24), 'bins_edad'] = 4
    df.loc[df['Age'] > 36, 'bins_edad'] = 5
    
    return df

# Aplicación
train = crear_bins_edad(train)
test = crear_bins_edad(test)

#### "tipo_raza"

Se usa las Variables de Raza 1 y Raza 2 (Breed1 y Breed2 respectivamente) para separar animales de raza pura de los que no lo son. Observamos en el EDA que un perro de raza tenderá a adoptarse más rápido que uno que no lo es. 

La logica del codigo es que si no hay segunda raza o si ambos valores son el mismo el animal es de raza pura. Si los dos valores son distintos es mestizo. Tiene en cuenta que si la raza ya es mezcla o genériza, la asigna como raza mezcla.

In [ ]:
razas_genericas = ['Mixed Breed', 'Domestic Short Hair', 'Domestic Medium Hair', 'Domestic Long Hair']

for df in [train, test]:
    # Tiene segunda raza
    tiene_breed2 = df['Breed2'] != 0
    
    # Breed1 es una raza genérica/mezclada (por label)
    breed1_generica = df['Breed1_label'].isin(razas_genericas)
    
    # Combinamos las tres condiciones como en el EDA
    df['tipo_raza'] = (tiene_breed2 | breed1_generica).astype(int)


#### Fetures de cantidad: "es_grupo" + "log_quantity" + "tipo_grupo"
 Crea dos variables a partir de Quantity:
- 'es_grupo': Binaria (1 si hay más de 1 animal, 0 si es individual).
- 'log_quantity': Suavizado logarítmico para mitigar outliers en cantidades grandes.
- "tipo_grupo": Como vimos en el EDA, separa las mascotas en Individual, grupo pequeño (2 o 3), o grupo grande (4 o más mascotas)

In [ ]:
def procesar_cantidad(df):
    df['es_grupo'] = 0
    df.loc[df['Quantity'] > 1, 'es_grupo'] = 1

    cantidades = df['Quantity'].fillna(1) 
    df['log_quantity'] = np.log1p(cantidades)

    def categorizar(q):
        if q == 1:
            return 0  # Individual
        elif q <= 3:
            return 1  # Grupo pequeño
        else:
            return 2  # Grupo grande

    df['tipo_grupo'] = df['Quantity'].apply(categorizar)
    
    return df

# Aplicación
train = procesar_cantidad(train)
test = procesar_cantidad(test)

#### Fetures de color: "num_colores" y "es_todo_negro"
- "num_colores" cuenta la cantidad de colores por animal
- "es_todo_negro" En muchos foros de Kaggle, leimos que muchos usuarios notaron un fenómeno real de la psicología humana: los perros y gatos completamente negros tardan más en ser adoptados (se lo conoce mundialmente como el "Black Dog Syndrome"). Creamos esta feature binaria.

Diccionario de colores dado por la competencia: 1 es Negro, 2 es Marron, 3 es Dorado, 4 es Amarillo, 5 es Crema, 6 es Gris, 7 es Blanco.

In [ ]:
# numero de colores
train['num_colores'] = (
    (train['Color1'] != 0).astype(int) +
    (train['Color2'] != 0).astype(int) +
    (train['Color3'] != 0).astype(int)
)

test['num_colores'] = (
    (test['Color1'] != 0).astype(int) +
    (test['Color2'] != 0).astype(int) +
    (test['Color3'] != 0).astype(int)
)

# Evitar contar colores repetidos
train['num_colores'] = train[['Color1','Color2','Color3']].replace(0, np.nan).nunique(axis=1)
test['num_colores'] = test[['Color1','Color2','Color3']].replace(0, np.nan).nunique(axis=1)

In [ ]:
def es_todo_negro(df):
    es_negro = lambda col: (df[col] == 1) | (df[col] == 0)
    
    df['es_todo_negro'] = (
        (df['Color1'] == 1) &
        es_negro('Color2') &
        es_negro('Color3')
    ).astype(int)
    
    return df

# Aplicación
train = es_todo_negro(train)
test = es_todo_negro(test)

train['es_todo_negro'] = ((train['Color1'] == 1) & (train['Color2'] == 0) & (train['Color3'] == 0)).astype(int)
test['es_todo_negro']  = ((test['Color1'] == 1)  & (test['Color2'] == 0)  & (test['Color3'] == 0)).astype(int)

#### "estado_agrupado"
Como vimos en el EDA la mayoría de casos se concentran en algunos Estados. Creamos una variable categórica que permita discriminar entre los más frecuentes y Otros.

In [ ]:
def crear_estado(df):
    df['estado_agrupado'] = 0  # Others
    df.loc[df['State_label'] == 'Selangor', 'estado_agrupado'] = 1
    df.loc[df['State_label'] == 'Kuala Lumpur', 'estado_agrupado'] = 2
    df.loc[df['State_label'] == 'Pulau Pinang', 'estado_agrupado'] = 3
    df.loc[df['State_label'] == 'Johor', 'estado_agrupado'] = 4
    df.loc[df['State_label'] == 'Perak', 'estado_agrupado'] = 5
    return df

# Aplicación
train = crear_estado(train)
test = crear_estado(test)

#### "tiene_video"

In [ ]:
def tiene_video(df):
    df['tiene_video'] = (df['VideoAmt'] > 0).astype(int)
    return df

# Aplicación
train = tiene_video(train)
test = tiene_video(test)

#### "tiene_foto"

In [ ]:
def tiene_foto(df):
    df['tiene_foto'] = (df['PhotoAmt'] > 0).astype(int)
    return df

# Aplicación
train = tiene_foto(train)
test = tiene_foto(test)

#### "bins_fotos"

Se busca discretizar la cantidad de fotos por animal en pocos rangos que sean amplios

- 0: sin fotos
- 1: pocas (1–3)
- 2: medias (4–7)
- 3: muchas (8+)

In [ ]:
def crear_bins_fotos(df):
    df['bins_fotos'] = 0  # default
    
    df.loc[df['PhotoAmt'] == 0, 'bins_fotos'] = 0
    df.loc[(df['PhotoAmt'] >= 1) & (df['PhotoAmt'] <= 3), 'bins_fotos'] = 1
    df.loc[(df['PhotoAmt'] >= 4) & (df['PhotoAmt'] <= 7), 'bins_fotos'] = 2
    df.loc[df['PhotoAmt'] >= 8, 'bins_fotos'] = 3
    
    return df

# Aplicación
train = crear_bins_fotos(train)
test = crear_bins_fotos(test)

#### "foto_por_animal"

In [ ]:
def foto_por_animal(df):
    # Evita división por cero en caso de que Quantity sea 0
    cantidad = df['Quantity'].replace(0, 1)
    df['foto_por_animal'] = df['PhotoAmt'] / cantidad
    return df

train = foto_por_animal(train)
test = foto_por_animal(test)

#### "grupo_sin_foto"

In [ ]:
train['grupo_sin_foto'] = ((train['Quantity'] > 1) & (train['PhotoAmt'] == 0)).astype(int)
test['grupo_sin_foto']  = ((test['Quantity'] > 1)  & (test['PhotoAmt'] == 0)).astype(int)

#### "es_gratis"

In [ ]:
# Binaria: 1 si la adopción es gratuita
train['es_gratis'] = (train['Fee'] == 0).astype(int)
test['es_gratis'] = (test['Fee'] == 0).astype(int)

#### "bins_fee"

Se busca discretizar la variable Fee en rangos amplios para que el modelo pueda capturar la no linealidad

- 0: gratis
- 1: bajo (entre 0 y 50)
- 2: medio (entre 50 y 200)
- 3: alto (mayor a 200)

In [ ]:
def crear_bins_fee(df):
    df['bins_fee'] = 0  # default
    
    df.loc[df['Fee'] == 0, 'bins_fee'] = 0
    df.loc[(df['Fee'] > 0) & (df['Fee'] <= 50), 'bins_fee'] = 1
    df.loc[(df['Fee'] > 50) & (df['Fee'] <= 200), 'bins_fee'] = 2
    df.loc[df['Fee'] > 200, 'bins_fee'] = 3
    
    return df

# Aplicación
train = crear_bins_fee(train)
test = crear_bins_fee(test)

#### "perfil_rescatista"
Categoriza a cada rescatista según su nivel de actividad según analizamos en el EDA.

In [ ]:
# ── Calcular publicaciones por rescatista en todo el dataset (df2) ──────────────────────────────
rescuer_publicaciones = df2.groupby('RescuerID')['PetID'].count().reset_index()
rescuer_publicaciones.columns = ['RescuerID', 'n_publicaciones']

# ── Categorizar perfil ────────────────────────────────────────────────────────
def categorizar_perfil(n):
    if n == 1:
        return 0  # Ocasional
    elif n <= 5:
        return 1  # Pequeño
    elif n <= 15:
        return 2  # Mediano
    else:
        return 3  # Alto volumen

rescuer_publicaciones['perfil_rescatista'] = rescuer_publicaciones['n_publicaciones'].apply(categorizar_perfil)

# ── Agregar a train y test ────────────────────────────────────────────────────
train = train.merge(rescuer_publicaciones[['RescuerID', 'perfil_rescatista']], on='RescuerID', how='left')
test  = test.merge(rescuer_publicaciones[['RescuerID', 'perfil_rescatista']], on='RescuerID', how='left')

#### "top_5_rescatistas"
Como vimos en el EDA, quién es el rescatista puede incluir en la tasa de adopción.

Esta variable permite identificar a los 5 rescatistas con más actividad en el train, para que en caso de aparecer en el test, aprenda de sus patrones de comportamiento y efectividad de adopción

In [ ]:
# ── Identificar top 5 rescatistas en train ────────────────────────────────────
top5 = (train.groupby('RescuerID')
        .agg(
            n_publicaciones=('PetID', 'count'),
            pct_adoptados=('AdoptionSpeed', lambda x: (x < 4).mean() * 100)
        )
        .sort_values('n_publicaciones', ascending=False)
        .head(5)
        .reset_index())

top5.index = range(1, 6)
print(top5[['RescuerID', 'n_publicaciones', 'pct_adoptados']])

# ── Crear feature ─────────────────────────────────────────────────────────────
rescuer_to_rank = {row['RescuerID']: rank for rank, row in top5.iterrows()}

train['top_5_rescatistas'] = train['RescuerID'].map(rescuer_to_rank).fillna(0).astype(int)
test['top_5_rescatistas']  = test['RescuerID'].map(rescuer_to_rank).fillna(0).astype(int)

Si se compara con el EDA se puede ver que los 5 rescatistas identificados en el dataset de train son los mismos 5 que en el dataset completo del EDA y su porcentaje de adopción prácticamente el mismo. Esto es bueno, ya que sustenta la premisa de que cada persona tiene un patron de comportamiento/personalidad/efectividad que está relacionada con la velocidad de adopción de las mascotas que publica.

### Otras combinaciones de variables
#### "sano_y_gratis"

In [ ]:
train['sano_y_gratis'] = ((train['Health'] == 1) & (train['Fee'] == 0)).astype(int)
test['sano_y_gratis']  = ((test['Health'] == 1)  & (test['Fee'] == 0)).astype(int)

#### "estado_salud_compuesto"

In [ ]:
# Para cada variable sanitaria, binarizamos si el valor es 1 (= Sí / Sano)
# y sumamos. El resultado va de 0 (ningún cuidado) a 4 (todos los cuidados).
def estado_salud_compuesto(df):
    vacunado     = (df['Vaccinated'] == 1).astype(int)
    desparasitado = (df['Dewormed'] == 1).astype(int)
    esterilizado = (df['Sterilized'] == 1).astype(int)
    sano         = (df['Health'] == 1).astype(int)
    df['estado_salud_compuesto'] = vacunado + desparasitado + esterilizado + sano
    return df

train = estado_salud_compuesto(train)
test = estado_salud_compuesto(test)

#### "vacunado_y_sano"

In [ ]:
# Combinación sanitaria más directa que estado_salud_compuesto
train['vacunado_y_sano'] = ((train['Vaccinated'] == 1) & (train['Health'] == 1)).astype(int)
test['vacunado_y_sano']  = ((test['Vaccinated'] == 1)  & (test['Health'] == 1)).astype(int)

#### "edad_x_tipo"

In [ ]:
# Interacción numérica entre edad y tipo de animal
# Permite que el modelo aprenda que "6 meses" significa cosas distintas
# para un perro que para un gato
train['edad_x_tipo'] = train['Age'] * train['Type']
test['edad_x_tipo'] = test['Age'] * test['Type']

#### "es_cachorro" + "cachorro sin nombre"

In [ ]:
# Es cachorro: Binaria: 1 si el animal tiene 3 meses o menos
train['es_cachorro'] = (train['Age'] <= 3).astype(int)
test['es_cachorro'] = (test['Age'] <= 3).astype(int)

train['cachorro_sin_nombre'] = ((train['Age'] <= 3) & (train['tiene_nombre'] == 0)).astype(int)
test['cachorro_sin_nombre']  = ((test['Age'] <= 3)  & (test['tiene_nombre'] == 0)).astype(int)

#### "adulto_sin_estirilizar"

In [ ]:
train['adulto_sin_esterilizar'] = ((train['Age'] > 12) & (train['Sterilized'] != 1)).astype(int)
test['adulto_sin_esterilizar']  = ((test['Age'] > 12)  & (test['Sterilized'] != 1)).astype(int)

#### "listing_completo"

In [ ]:
# Si tiene foto y descripcion
# NO se aplica el log a "largo_descripcion" porque ya fue calculado (VER seccion 4.1.2)
train['listing_completo'] = ((train['PhotoAmt'] > 0) & (train['largo_descripcion'] > 0)).astype(int)
test['listing_completo']  = ((test['PhotoAmt'] > 0)  & (test['largo_descripcion'] > 0)).astype(int)


#### "gato_pelo_largo"

In [ ]:
# En perros el pelo largo no discrimina tanto
train['gato_pelo_largo'] = ((train['Type'] == 2) & (train['FurLength'] == 3)).astype(int)
test['gato_pelo_largo']  = ((test['Type'] == 2)  & (test['FurLength'] == 3)).astype(int)

## 4.3. Selección final de Features

Las variables que no son mencionadas o que presentaban una contribución marginal en términos de gain (como Health, tiene_video o es_todo_negro) fueron excluidas, resultando en un set final de 22 predictores optimizados para el entrenamiento del modelo LightGBM.

La lógica de selección se estructuró bajo los siguientes criterios:

### Filtrado por naturaleza y tipo de dato:

- Se excluyeron las variables de tipo **"label"** (etiquetas de texto originales) y los identificadores únicos como **PetID, RescuerID, Name y Description**, ya que no aportan capacidad de generalización al algoritmo.

- Se descartaron las variables de texto libre, priorizando sus derivaciones numéricas o categóricas ya procesadas.

### Tratamiento de familias de variables:
Para evitar la redundancia, en aquellos grupos de variables que parten de una misma raíz original, se seleccionaron únicamente las transformaciones más eficientes:

- **Grupo Quantity:** Se optó por tipo_grupo y log_quantity, descartando la variable cruda por estar mejor representada en la escala logarítmica y la categoría derivada.

- **Grupo Video y Foto:** Se priorizaron las variables binarias y de ratio como tiene_video, tiene_foto y foto_por_animal, eliminando las cantidades crudas que presentaban valores aislados.

- **Grupo Fee y Estado:** Se mantuvieron las versiones agrupadas y binarias (es_gratis, bins_fee, estado_agrupado) para reducir la alta cardinalidad de las variables originales y mejorar la estabilidad del modelo.

In [ ]:
# Definición unificada de columnas a excluir tras el análisis de correlación e importancia
cols_excluir = (
    # 1. Variables de texto, IDs y etiquetas originales
    [col for col in train.columns if col.endswith('_label')] +
    ['Name', 'Description', 'RescuerID', 'PetID'] +
    
    # 2. Variables originales reemplazadas por versiones procesadas
    ['Quantity', 'es_grupo',    # Representadas por log_quantity y tipo_grupo
     'PhotoAmt', 'VideoAmt',    # Representadas por foto_por_animal, tiene_foto y tiene_video
     'Fee',                     # Representadas por es_gratis y bins_fee
     'State'] +                 # Representada por estado_agrupado
    
    # 3. Variables con alta colinealidad o baja importancia (Resultado del segundo filtrado)
    ['log_quantity', 'bins_fee', 'listing_completo']
)

# Aplicar la eliminación en ambos datasets
train = train.drop(columns=cols_excluir)
test  = test.drop(columns=cols_excluir)

# Verificación de dimensiones finales
print(f"Dimensiones finales - Train: {train.shape}")
print(f"Dimensiones finales - Test: {test.shape}")

### Refinamiento por Correlación:

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))

corr = train.drop(columns='AdoptionSpeed').corr(method='spearman', numeric_only=True)

sns.heatmap(corr, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            center=0,
            linewidths=0.5,
            annot_kws={'size': 6},
            ax=ax)

ax.set_title('Correlación de Spearman entre variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Top 10 correlaciones más fuertes ─────────────────────────────────────────
corr_pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
                  .stack()
                  .reset_index())
corr_pairs.columns = ['Variable 1', 'Variable 2', 'Correlacion']
corr_pairs['Correlacion_abs'] = corr_pairs['Correlacion'].abs()

top10 = corr_pairs.sort_values('Correlacion_abs', ascending=False).head(10)
top10 = top10.drop(columns='Correlacion_abs').reset_index(drop=True)
top10.index += 1

print(top10.to_string())


- **Edad y Ciclo de Vida:** Se detectó una correlación muy alta (0.95) entre Age y edad_x_tipo, así como con es_cachorro (-0.87). Se optó por mantener únicamente Age como variable continua original, ya que contiene la información más granular y evita la redundancia de las categorías derivadas.

- **Salud:** Variables como Vaccinated, Dewormed y vacunado_y_sano presentaron correlaciones superiores a 0.81 con estado_salud_compuesto. Se decidió conservar esta última junto con Sterilized, simplificando la señal del estado sanitario en una variable integrada de mayor peso predictivo.

- **Fee e Interacciones:** Dada la fuerte correlación (0.90) entre es_gratis y sano_y_gratis, se seleccionó la interacción sano_y_gratis. Esta elección permite capturar no solo el factor económico, sino también la condición de salud, lo cual impacta directamente en la velocidad de adopción.

- Demas Features:

    - **Colores:** Ante la correlación de 0.82 entre Color3 y num_colores, se mantuvo Color3 para preservar la información específica de la combinación de colores.

    - **Identificación:** Se descartó cachorro_sin_nombre en favor de tiene_nombre (correlación -0.79), al ser un atributo más general y menos sesgado por la edad.

    - **Fotos:** Se priorizó foto_por_animal y tiene_foto para modelar la presencia y cantidad de material visual sin incurrir en colinealidad con el conteo bruto original.

### Features Finales:

In [ ]:
features = [
    # Feature Tipo
    'Type',

    # Feature Género
    'Gender',
    
    # Features de tamaño y pelaje
    'MaturitySize',
    'FurLength',
    

    # Features creadas ea partir de texto
    'tiene_nombre',
    'largo_descripcion',

    # Features de Edad
    'Age',                   # Variable original

    # Features de raza
    'tipo_raza',
    'Breed1',               # Variable original
    'Breed2',               # Variable original

    # Features de cantidad
    'tipo_grupo',

    # Features de colores
    'Color1',                  # Variable original
    'Color2',                  # Variable original
    'Color3',                  # Variable original

    # Features de Estado
    'estado_agrupado',

    #Features de Foto
    'tiene_foto',
    'foto_por_animal',

    # Features de Rescatistas
    'perfil_rescatista',
    'top_5_rescatistas',

    # Features de salud
    'Sterilized',           # Variable original
    'estado_salud_compuesto',

    # Interacciones
    'sano_y_gratis',
    'gato_pelo_largo'
]


label = 'AdoptionSpeed'

In [ ]:
# Aclaro cuales features son categóricas
# Para que LightGBM las trate correctamente sin necesidad de hacer One-Hot Encoding manual

categorical_features = [
    'Type', 'Gender', 'MaturitySize', 'FurLength',
    'tipo_raza', 'Breed1', 'Breed2',
    'tipo_grupo', 'Color1', 'Color2', 'Color3',
    'estado_agrupado',
    'Sterilized',
    'perfil_rescatista', 'top_5_rescatistas',
    'tiene_foto',
    'tiene_nombre',
    'estado_salud_compuesto',
    'sano_y_gratis',
    'gato_pelo_largo'
]


---
# Sección 5. LightGBM + Optimización con Optuna


In [ ]:
#Genero dataframes de train y test con sus respectivos targets
X_train = train[features]
y_train = train[label]

X_test = test[features]
y_test = test[label]

In [ ]:
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_TEMP_FILES)

### Hiperparámetros a optimizar en Optuna

In [ ]:
def lgb_custom_metric_kappa(dy_pred, dy_true):
    metric_name = 'kappa'
    value = cohen_kappa_score(dy_true.get_label(),dy_pred.argmax(axis=1),weights = 'quadratic')
    is_higher_better = True
    return(metric_name, value, is_higher_better)

def cv_es_lgb_objective(trial):
    
    lgb_params = {      
                    'objective':        'multiclass',
                    'verbosity':        -1,
                    'num_class':        len(y_train.unique()),
                    'seed':             SEED,
                    'deterministic':    True,
                        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
                        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
                        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
                        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
                        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
                        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
                        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                        } 


    # Estimaciones de los 5 modelos del CV sobre los datos test y los acumulo en la matriz scores_ensemble
    scores_ensemble = np.zeros((len(y_test),len(y_train.unique())))

    # Score del 5 fold CV inicializado en 0
    score_folds = 0

    # Numero de splits del CV
    n_splits = 5 # Se deja como el Baseline 

    # Objeto para hacer el split estratificado de CV
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for i, (if_index, oof_index) in enumerate(skf.split(X_train, y_train)):
        
        # Dataset in fold (donde entreno) 
        lgb_if_dataset = lgb.Dataset(data=X_train.iloc[if_index],
                                        label=y_train.iloc[if_index],
                                        categorical_feature=categorical_features,
                                        free_raw_data=False)
        
        # Dataset Out of fold (donde mido la performance del CV)
        lgb_oof_dataset = lgb.Dataset(data=X_train.iloc[oof_index],
                                        label=y_train.iloc[oof_index],
                                        categorical_feature=categorical_features,
                                        free_raw_data=False)

        # Entreno el modelo
        lgb_model = lgb.train(lgb_params,
                                lgb_if_dataset,
                                valid_sets=lgb_oof_dataset,
                                callbacks=[lgb.early_stopping(10, verbose=False)],
                                feval = lgb_custom_metric_kappa
                                )
        
        # Acumulo los scores (probabilidades) de cada clase para cada uno de los modelos que determino en los folds
        # Se predice el 20% de los datos que separe para tes y no uso para entrenar en ningun fold
        scores_ensemble = scores_ensemble + lgb_model.predict(X_test)
        
        #Score del fold (registros de dataset train que en este fold quedan out of fold)
        score_folds = score_folds + cohen_kappa_score(y_train.iloc[oof_index], 
                                                            lgb_model.predict(X_train.iloc[oof_index]).argmax(axis=1),weights = 'quadratic')/n_splits


    # Guardo prediccion del trial sobre el conjunto de test
    # Genero nombre de archivo
    predicted_filename = os.path.join(PATH_TO_TEMP_FILES,f'test_{trial.study.study_name}_{trial.number}.joblib')
    # Copia del dataset para guardar la prediccion
    predicted_df = test.copy()
    # Genero columna pred con predicciones sumadas de los 5 folds
    predicted_df['pred'] = [scores_ensemble[p,:] for p in range(scores_ensemble.shape[0])]
    # Grabo dataframe en temp_artifacts
    dump(predicted_df, predicted_filename)
    # Indico a optuna que asocie el archivo generado al trial
    upload_artifact(trial, predicted_filename, artifact_store)    


    # ----------------------------

    cm_filename = os.path.join(PATH_TO_TEMP_FILES, f'cm_{trial.study.study_name}_{trial.number}.png')

    # Generar la figura con la función de utils
    fig = plot_confusion_matrix(y_test, scores_ensemble.argmax(axis=1))

    # Forzar el color de los números internos a NEGRO
    fig.update_annotations(font=dict(color='black'))

    # 4. CONFIGURACIÓN DE COLORES Y FONDO:
    fig.update_layout(
        template='plotly_white', 
        paper_bgcolor='white',   
        plot_bgcolor='white',    
        font=dict(color='black') 
    )
    
    # Cambiar escala a Rojo-Amarillo-Verde (RdYlGn)
    # Esta escala pone el rojo en los valores bajos y el verde en los altos
    fig.update_traces(colorscale='RdYlGn', selector=dict(type='heatmap'))

    # Grabo archivo usando el motor kaleido
    fig.write_image(cm_filename, engine="kaleido")

    # Asocio al trial
    upload_artifact(trial, cm_filename, artifact_store)

    # -------------------------------

    # Determino score en conjunto de test y asocio como metrica adicional en optuna
    test_score = cohen_kappa_score(y_test,scores_ensemble.argmax(axis=1),weights = 'quadratic')
    trial.set_user_attr("test_score", test_score)

    # Devuelvo score del 5fold cv a optuna para que optimice en base a eso
    return(score_folds)

## 5.1 Optuna

In [ ]:
# ✅ Reemplazá la celda de la sección 5.1 con esto:

artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)

# Usamos la ruta absoluta que ya fue creada arriba
db_path = OPTUNA_DIR / "db.sqlite3"

study = optuna.create_study(
    direction='maximize',
    storage=f"sqlite:///{db_path}",   # ← ruta absoluta, sin ../
    study_name="EDA&Tabulares_pruebaEnsemble",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

study.optimize(cv_es_lgb_objective, n_trials=100)

## 5.2  Importancia de Features — Promedio de los 5 modelos del CV

In [ ]:
# Tomamos los mejores hiperparámetros que encontró Optuna
mejores_params = {
    'objective':     'multiclass',
    'verbosity':     -1,
    'num_class':     len(y_train.unique()),
    'seed':          SEED,
    'deterministic': True
} | study.best_params

# Acumulador de importancias — una fila por fold
importancias_folds = np.zeros(len(features))

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

for i, (if_index, oof_index) in enumerate(skf.split(X_train, y_train)):
    
    lgb_if_dataset = lgb.Dataset(data=X_train.iloc[if_index],
                                  label=y_train.iloc[if_index],
                                  free_raw_data=False)
    
    lgb_oof_dataset = lgb.Dataset(data=X_train.iloc[oof_index],
                                   label=y_train.iloc[oof_index],
                                   free_raw_data=False)
    
    modelo_fold = lgb.train(mejores_params,
                            lgb_if_dataset,
                            valid_sets=lgb_oof_dataset,
                            callbacks=[lgb.early_stopping(10, verbose=False)],
                            feval=lgb_custom_metric_kappa)
    
    # Sumamos la importancia de este fold al acumulador
    importancias_folds += modelo_fold.feature_importance(importance_type='gain')

# Promediamos dividiendo por la cantidad de folds
importancias_promedio = importancias_folds / n_splits

# Armamos el DataFrame final ordenado de mayor a menor
importance = pd.DataFrame({
    'feature':    features,
    'importance': importancias_promedio
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(importance.to_string())


# Configuramos el tamaño del gráfico
plt.figure(figsize=(10, 8))

# Graficamos el Top 20 de tu DataFrame ya promediado
sns.barplot(x='importance', y='feature', data=importance.head(20), palette='viridis')

plt.title('Feature Importance (Promedio de 5 Folds CV)', fontsize=14, pad=15)
plt.xlabel('Importancia Promedio', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.tight_layout()
plt.show()

## 5.3 Matriz de Correlacion de Spearman

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))

corr = X_train.corr(method='spearman')

sns.heatmap(corr, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            center=0,
            linewidths=0.5,
            annot_kws={'size': 8},
            ax=ax)

ax.set_title('Correlación de Spearman entre variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Top 10 correlaciones más fuertes ─────────────────────────────────────────
corr_pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
                  .stack()
                  .reset_index())
corr_pairs.columns = ['Variable 1', 'Variable 2', 'Correlacion']
corr_pairs['Correlacion_abs'] = corr_pairs['Correlacion'].abs()

top10 = corr_pairs.sort_values('Correlacion_abs', ascending=False).head(10)
top10 = top10.drop(columns='Correlacion_abs').reset_index(drop=True)
top10.index += 1

print(top10.to_string())

## 5.4 Resultados Finales

In [ ]:


# El mejor trial (intento) encontrado por Optuna
mejor_trial = study.best_trial

# Extraemos los valores
kappa_cv = mejor_trial.value
kappa_test = mejor_trial.user_attrs.get("test_score")
gap = kappa_cv - kappa_test

# 1. Métricas Principales
print(f" Mejor Kappa en CV (Cross-Validation): {kappa_cv:.4f}")
print(f" KAPPA FINAL EN TEST SET: {kappa_test:.4f}")
print(f" GAP (CV - Test): {gap:.4f}")

# 2. Análisis Automático del Gap
print("\n Diagnóstico del Modelo:")
if gap > 0.05:
    print(" DIAGNÓSTICO: CV >> Test → Overfitting.")
    print("   El modelo está memorizando el Train. Deberías aumentar la regularización.")
elif abs(gap) <= 0.05:
    print(" DIAGNÓSTICO: CV ≈ Test → Buena Generalización.")
    print("   El modelo aprendió patrones que se mantienen en datos nuevos.")
else:
    print(" DIAGNÓSTICO: CV < Test → Posible Data Leaking.")
    print("   Es inusual que el test dé mucho mejor que el CV. Revisar si hay filtración de datos.")

# 3. Los parámetros ganadores
print("\n Mejores Hiperparámetros:")
for clave, valor in mejor_trial.params.items():
    print(f"    {clave}: {valor}")


print("Study:", study.study_name)
print("Best trial:", mejor_trial.number)

---
# 6 Guardado de la mejor prediccion para Ensemble

In [ ]:
# Guardar predicciones del MEJOR trial para posterior ensemble──

mejores_params = {
    'objective':     'multiclass',
    'verbosity':     -1,
    'num_class':     len(y_train.unique()),
    'seed':          SEED,
    'deterministic': True
} | study.best_params

scores_ensemble = np.zeros((len(y_test), len(y_train.unique())))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for i, (if_index, oof_index) in enumerate(skf.split(X_train, y_train)):
    lgb_if_dataset = lgb.Dataset(
        data=X_train.iloc[if_index],
        label=y_train.iloc[if_index],
        categorical_feature=categorical_features,
        free_raw_data=False
    )
    lgb_oof_dataset = lgb.Dataset(
        data=X_train.iloc[oof_index],
        label=y_train.iloc[oof_index],
        categorical_feature=categorical_features,
        free_raw_data=False
    )
    lgb_model = lgb.train(
        mejores_params,
        lgb_if_dataset,
        valid_sets=lgb_oof_dataset,
        callbacks=[lgb.early_stopping(10, verbose=False)],
        feval=lgb_custom_metric_kappa
    )
    scores_ensemble += lgb_model.predict(X_test)

# Mismo formato que usan los joblibs de los trials
best_predicted_df = test.copy()
best_predicted_df['pred'] = [scores_ensemble[p, :] for p in range(scores_ensemble.shape[0])]

output_path = os.path.join(WORK_DIR, 'best_lgb_predictions.joblib')
dump(best_predicted_df, output_path)

print(f" Mejor trial : #{study.best_trial.number} con Kappa CV: {kappa_cv:.4f} y Kappa Test: {kappa_test:.4f}")
print(f" Archivo llamado best_lgb_predictions.joblib listo para el ensemble. Guardado en : {output_path}")